In [ ]:
import pandas as pd

import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import re 
from tqdm import tqdm 
from rapidfuzz import fuzz, process
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from datetime import datetime
from collections import Counter
from collections import defaultdict
from itertools import product
import os 

In [ ]:
# MODULE
def clean_name(text):
    if pd.isna(text): 
        return text 
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def set_value(gdf):
    gdf['category'] = None
    gdf['sub_category'] = None
    gdf['type'] = None
    gdf['score'] = 0.0
    gdf['log'] = None
    return gdf

def to_list(config_file, config_column):
    df = config_file

    if isinstance(config_column, str) : config_column = [config_column]
    result = [df[col].dropna().tolist() for col in config_column]

    if len(result) == 1 : return result[0]
    return result

def to_dict(config_file, key_column, value_column):
    key = config_file[key_column].dropna().tolist()
    value = config_file[value_column].dropna().tolist() 

    dictionary = defaultdict(list)
    for k, v in zip(key, value):
        dictionary[k].append(v)
    
    return dict(dictionary)

def refine_crs(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    gdf = gdf.to_crs(epsg=3857)

    if gdf.crs is None: 
        print(f' Warning Massage : There is a problem with GDF CRS')

    return gdf

def describe_gdf(gdf):
    print('GDF Description')
    print(f'---------------')

    try: print(f'📦 Total Records : {len(gdf)}')
    except: print('⚠️ Cannot Count GDF Reccord')

    try: print(f'🧱 Columns : {list(gdf.columns)}')
    except: print('⚠️ Cannot Show GDF Reccord')
    
    if gdf.crs : print(f'🌍 CRS : {gdf.crs}') 
    else : print('🌍 CRS : None')

    if 'geometry' in gdf : geom_types = gdf.geom_type.unique()
    else : 'None'
    print(f'🧭 Geometry Type: {geom_types}')

    if 'geometry' in gdf : print(f'📍 Missing geometry: {(gdf.geometry.isna()).sum()}')
    else: print(f'📍 Missing geometry: 0')
    
    print('---------------')
    try:
        gdf = gdf[gdf.geometry.notna()]
        gdf = gdf.to_crs(epsg=4326) if gdf.crs else gdf.set_crs(epsg=4326)
        gdf['lon'] = gdf.geometry.x
        gdf['lat'] = gdf.geometry.y

        print(f"Longitude range: {gdf['lon'].min():.2f} – {gdf['lon'].max():.2f}")
        print(f"Latitude range: {gdf['lat'].min():.2f} – {gdf['lat'].max():.2f}")

        if gdf['lon'].between(95, 141).mean() < 0.95 or gdf['lat'].between(-11, 6).mean() < 0.95:
            print("⚠️ Some coordinates are outside Indonesia bounds.")
        else:
            print("✅ All points within Indonesia bounds.")
        print()
    except:
        print('⚠️ Cannot Process Information of Coordinate')

def remove_stopword(word, stopword_list):
    word_low = str(word).lower()
    for stopword in stopword_list:
        word_low = word_low.replace(stopword, '')
    return ' '.join(word_low.split()).strip()
    
def correct_typo_token(word, candidates, threshold=70):
    best_match, score, _ = process.extractOne(word, candidates, scorer = fuzz.token_set_ratio)
    if score >= threshold:
        return best_match, score
    else:
        return "other", score

def correct_typo_dict(word, candidates, threshold=70):
    flat = {alias: key for key, aliases in candidates.items() for alias in aliases}
    all_aliases = list(flat.keys())

    best_match, score = correct_typo_token(word, all_aliases, threshold)
    if score > threshold:
        return flat[best_match], score
    else: 
        return 'other', score

def validate_subclass(subclass_dict):
    warn = False
    validate = subclass_dict.copy()

    for name, val in subclass_dict.items():
        if not isinstance(val, bool):
            if not warn:
                print("⚠️ Warning Message! All subclass parameters must be of type bool (True/False)")
                warn = True
            print(f"⚠️ Warning Message! Proceed {name} to False")
            validate[name] = False
    
    return validate

def generate_result(master_list, len_gdf):
    count_dict = {item[0] : item[2] for item in master_list}
    sub_names = list(count_dict.keys())
    result_dict = {}

    for combination in product([True, False], repeat = len(sub_names)):
        active = [name for name, is_active in zip(sub_names, combination) if is_active]
        classified_total = sum(count_dict[name] for name in active)
        unclassified_total = len_gdf -classified_total

        class_result = {
            sub_name : (count_dict[sub_name] if sub_name in active else 0)
            for sub_name in sub_names
        }
        class_result['classified'] = classified_total
        class_result['unclassified'] = unclassified_total
        result_dict[tuple(combination)] = class_result
    
    return result_dict

def merge_selected(master_list, combinations=None):
    categories = [item[0] for item in master_list]

    if combinations is None:
        all_combinations = product([True, False], repeat=len(categories))
        combinations = {}

        for combo in all_combinations:
            gdf_list = [item[1] for item, active in zip(master_list, combo) if active]
            combinations[combo] = gdf_list
    
    merged_result = {}
    for combination, gdf_list in combinations.items():
        if isinstance(gdf_list, list) and len(gdf_list) > 0:
            merged_result[combination] = gpd.GeoDataFrame(
                pd.concat(gdf_list, ignore_index=True), geometry='geometry', crs =gdf_list[0].crs 
            )
        else:
            merged_result[combination] = None

    return merged_result

def geodeduplicate(gdf, eps=100):
    gdf = gdf[gdf.geometry.notna()].copy()
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    gdf = gdf.to_crs(epsg=3857)

    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))

    db = DBSCAN(eps=eps, min_samples=1).fit(coords) 
    gdf['cluster'] = db.labels_

    gdf = gdf.sort_values('review', ascending=False)
    gdf = gdf.drop_duplicates(subset='cluster', keep='first')
    gdf = gdf.drop(columns='cluster')
    
    return gdf

def advance_geodeduplicate(
    gdf,
    cleaned_name_column,
    review_column_name,
    eps = 500,
    min_samples = 1,
    sim_treshold = 75,
    use_tfidf = False,
    log = False
    ):
    
    if len(gdf) == 0 : return gdf
    
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf.reset_index(drop=True)

    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
    db = DBSCAN (eps = eps, min_samples = min_samples).fit(coords)
    gdf['geocluster'] = db.labels_
    dedups_ids = {}

    if use_tfidf:
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(gdf[cleaned_name_column].astype(str))
        feature_names = np.array(vectorizer.get_feature_names_out())

    for cluster_id, cluster_gdf in gdf.groupby('geocluster'):
        names = cluster_gdf[cleaned_name_column].astype(str).tolist()
        cluster_map = {}
        processed = set()
        curent_group = 0

        for name in names:
            if name in processed:
                continue 

            cluster_map[name] = curent_group
            matches = process.extract(
                name, names, scorer=fuzz.token_set_ratio, score_cutoff=sim_treshold
                )
            
            for match_name, score, _ in matches:
                cluster_map[match_name] = curent_group
                processed.add(match_name)
                
            processed.add(name)
            curent_group += 1

            if use_tfidf:
                cluster_indices = cluster_gdf.index
                cluster_tfidf = tfidf_matrix[cluster_indices, :]
                mean_tfidf = np.asarray(cluster_tfidf.mean(axis=0)).ravel()

                top_anchor_idx = mean_tfidf.argsort()[-3:][::-1]
                anchors_tfidf = feature_names[top_anchor_idx]
                
                tokens = [re.findall(r'\b\w+\b', n.lower()) for n in names]
                flat = [t for sub in tokens for t in sub]
                freq = Counter(flat)
                anchors_counter = [t for t, c in freq.items() if c >= 2 and len (t) > 3]

                anchors = set(list(anchors_tfidf) + anchors_counter)

                for anchor in anchors:
                    same_anchor = [n for n in names if anchor in n.lower()]
                    if len(same_anchor) > 1:
                        for n in same_anchor:
                            cluster_map[n] = f'anchor_{anchor}'

        for idx, name in zip(cluster_gdf.index, names):
            dedups_ids[idx] = f'{cluster_id}_{cluster_map[name]}'

    gdf['dedup_id'] = gdf.index.map(dedups_ids)
    gdf = gdf.sort_values(review_column_name, ascending=False)

    if log:
        cluster_size = gdf.groupby('dedup_id')['review'].transform('size')
        max_review = gdf.groupby('dedup_id')['review'].transform('max')
        
        gdf['cluster_log'] =  None
        mask_accepted = (cluster_size > 1) & (gdf['review'] == max_review)
        mask_rejected = (cluster_size > 1) & (gdf['review'] != max_review)

        gdf.loc[cluster_size == 1, 'cluster_log'] = 'Non-cluster'
        gdf.loc[mask_accepted, 'cluster_log'] = (
            gdf.loc[mask_accepted, 'dedup_id'].astype(str) + ' Accepted ✅'
        )
        gdf.loc[mask_rejected, 'cluster_log'] = (
            gdf.loc[mask_rejected, 'dedup_id'].astype(str) + ' Rejected ⚠️'
        )

        return gdf

    gdf = gdf.drop_duplicates(subset='dedup_id', keep='first')
    gdf = gdf.drop(columns=['geocluster', 'dedup_id'], errors='ignore')
    return gdf

def match_brand(name, brand_dict, threshold=70):
    keywords_to_brand = {
        alias : brand
        for brand, aliases in brand_dict.items()
        for alias in aliases
    }
    keyword_list = list(keywords_to_brand.keys())

    best_match, score, _ = process.extractOne(
        name,
        keyword_list,
        scorer=fuzz.token_set_ratio
    )
    
    if best_match and score > threshold:
        return best_match, score, keywords_to_brand.get(best_match)
    else:
        return None, score, None

In [ ]:
config = pd.read_csv('configuration.csv', sep=';')

# BANKING CONFIGURATION
BANKING_SUB, BANKING_POTENTIAL, BANKING_NOISE = to_list(config, 
    ['BANKING_SUB', 'BANKING_POTENTIAL', 'BANKING_NOISE'])

BANKING_ATM_KEY = to_list(config, 'BANKING_ATM_KEY')

BANKING_BANK_START, BANKING_BANK_BI = to_list(config, ['BANKING_BANK_START', 'BANKING_BANK_BI'])

BANKING_BANK_BRAND = to_dict(config, 'BANKING_BANK_BRAND_K', 'BANKING_BANK_BRAND_V')

# FINANCIAL SERVICE CONFIGURATION 

FIN_SUB = to_list(config, 'FIN_SUB')

LIFE_KEY = to_list(config, 'LIFE_KEY')

COP_KEY = to_list(config, 'COP_KEY')

PAWN_KEY = to_list(config, 'PAWN_KEY')

# HEALTH CONFIGURATION
HEALTH_SUB, HEALTH_NOISE,= to_list(config,
    ['HEALTH_SUB', 'HEALTH_NOISE'])

HEALTH_HOSPITAL_KEY_1, HEALTH_HOSPITAL_KEY_2, HEALTH_HOSPITAL_STOPWORD = to_list(config, 
    ['HEALTH_HOSPITAL_KEY_1', 'HEALTH_HOSPITAL_KEY_2', 'HEALTH_HOSPITAL_STOPWORD'])

HEALTH_CLINIC_KEY = to_list(config, 'HEALTH_CLINIC_KEY')

HEALTH_PHARMACY_SUB, HEALTH_PHARMACY_KEY, HEALTH_PHARMACY_BRAND = to_list(config, 
    ['HEALTH_APOTEK_SUB', 'HEALTH_APOTEK_KEY', 'HEALTH_APOTEK_BRAND'])

# SCHOOL CONFIGURATION
SCHOOL_SUB, SCHOOL_NOISE = to_list(config, ['SCHOOL_SUB','SCHOOL_NOISE'])

SCHOOL_SWASTA_KEY, SCHOOL_NEGERI_KEY = to_list(config, 
    ['SCHOOL_SWASTA_KEY', 'SCHOOL_NEGERI_KEY'])

SCHOOL_SWASTA_BRAND = to_dict(config, 'SCHOOL_SWASTA_BRAND_K', 'SCHOOL_SWASTA_BRAND_V')

SCHOOL_TK_KEY, SCHOOL_SD_KEY, SCHOOL_SMP_KEY, SCHOOL_SMA_KEY, SCHOOL_INTER_KEY = to_list(config,
    ['SCHOOL_TK_KEY', 'SCHOOL_SD_KEY', 'SCHOOL_SMP_KEY', 'SCHOOL_SMA_KEY', 'SCHOOL_INTER_KEY'])

SCHOOL_SPECIAL_KEY, SCHOOL_NONFORMAL_KEY, SCHOOL_UNIT_KEY = to_list(config, 
    ['SCHOOL_SPECIAL_KEY', 'SCHOOL_NONFORMAL_KEY', 'SCHOOL_UNIT_KEY'])

# HIGHER EDUCATION CONFIGURATION

UNIV_SUB, UNIV_NOISE, UNIV_SUB_NOISE, UNIV_ENTRY, UNIV_STOPWORDS = to_list(config, 
    ['UNIV_SUB', 'UNIV_NOISE', 'UNIV_SUB_NOISE', 'UNIV_ENTRY','UNIV_STOPWORDS'])

UNIV_POTENTIAL = to_dict(config, 'UNIV_POTENTIAL_K', 'UNIV_POTENTIAL_V')

UNIV_SWASTA_BRAND = to_dict(config, 'UNIV_SWASTA_BRAND_K', 'UNIV_SWASTA_BRAND_V')

# RETAIL & SHOPPING CONFIGURATION

STORE_SUB, STORE_NOISE, STORE_START_NOISE = to_list(config,
    ['STORE_SUB', 'STORE_NOISE', 'STORE_START_NOISE'])

STORE_MINIMARKET_KEY, STORE_SUPERMARKET_KEY, STORE_MALL_KEY = to_list(config, 
    ['STORE_MINIMARKET_KEY', 'STORE_SUPERMARKET_KEY', 'STORE_MALL_KEY'])

STORE_SUPERMARKET_BRAND = to_dict(config, 'STORE_SUPERMARKET_BRAND_K', 'STORE_SUPERMARKET_BRAND_V')

STORE_MALL_BRAND = to_dict(config, 'STORE_MALL_BRAND_K', 'STORE_MALL_BRAND_V')

# HOSPITALITY & ACCOMODATION CONFIGURATION

HOSPITALITY_SUB, HOSPITALITY_NOISE = to_list(config, ['HOSPITALITY_SUB', 'HOSPITALITY_NOISE'])

HOSPITALITY_VILLA_KEY, HOSPITALITY_HOTEL_KEY, HOSPITALITY_GUEST_KEY, HOSPITALITY_BOARD_KEY = to_list(config,
    ['HOSPITALITY_VILLA_KEY', 'HOSPITALITY_HOTEL_KEY', 'HOSPITALITY_GUEST_KEY', 'HOSPITALITY_BOARD_KEY'])

# INDUSTRY CONFIGURATION 

INDUSTRY_OFFICE_KEY, INDUSTRY_SUB = to_list(config, ['INDUSTRY_OFFICE_KEY', 'INDUSTRY_SUB'])

INDUSTRY_WAREHOUSE_KEY = to_list(config, 'INDUSTRY_WAREHOUSE_KEY')

INDUSTRY_FACTORY_KEY = to_list(config, 'INDUSTRY_FACTORY_KEY')

# FOOD AND BEVERAGE CONFIGURATION


In [ ]:
value = BANKING_SUB
value = ','.join(value)
value

In [ ]:
func = 'name contain value1;name start with value1;name_simillarity_treshold;subclass contain value1;subclass start with value1;review more than value 1;review less than value1'
func2 = func.replace('1', '2')
func3 = func.replace('1', '3')
func2


In [ ]:
brand = BANKING_BANK_BRAND.copy()

brand_new = {}
for key, values in brand.items():
    merged_values = ','.join(values)
    new_values = merged_values
    brand_new[key] = new_values

new_val = brand_new.keys()
new_val = "', '".join(new_val)
new_val = new_val.replace("', '", ";")
# new_val = 
# new_val = new_val.replace("', '", ";")
# # 

new_val

In [ ]:
name_similarity_treshold =

In [ ]:
BANKING_SUB = BANKING
new_banking_sub = old.replace(',', '-')

In [ ]:
# MODULE POI 

def poi_bank(
    gdf_bank,
    cleaned_name_column,
    subclass_column,
    verbose = True,
    atm = True,
    bank = True,
    micro = True
    ):
    
    print('⚙️ Processing Banking Category Point of Interest (POI) Filter Object')
    gdf_bank[cleaned_name_column] = gdf_bank[cleaned_name_column].apply(clean_name)
    gdf_bank[subclass_column] =  gdf_bank[subclass_column].apply(clean_name)
    gdf_ = len(gdf_bank)

    atm, bank, micro = validate_subclass({'atm' : atm, 'bank' : bank, 'micro' : micro}).values()

    for index, row in tqdm(gdf_bank.iterrows(), total = len(gdf_bank)):
        name = row[cleaned_name_column]
        sub = str(row[subclass_column])
        best_match, score, brand_name = match_brand(name, BANKING_BANK_BRAND, threshold=70)

        if any(potential in name for potential in BANKING_ATM_KEY):
            if any(potential in sub for potential in BANKING_SUB):
                if score > 75:
                    gdf_bank.at[index, 'category'] = 'Banking'
                    gdf_bank.at[index, 'sub_category'] = 'ATM'
                    gdf_bank.at[index, 'type'] = brand_name
                    gdf_bank.at[index, 'score'] = score
                    gdf_bank.at[index, 'log'] = f'✅ | Contain ATM | Match Subclass | Match with {brand_name}'

                else:
                    gdf_bank.at[index, 'category'] = 'Banking'
                    gdf_bank.at[index, 'sub_category'] = 'ATM'
                    gdf_bank.at[index, 'type'] = 'other'
                    gdf_bank.at[index, 'score'] = score
                    gdf_bank.at[index, 'log'] = '✅ | Contain ATM | Match Sucbclass | Score < 75 Local ATM Indicate'
            
            else:
                if score > 75:
                    gdf_bank.at[index, 'category'] = 'Banking'
                    gdf_bank.at[index, 'sub_category'] = 'ATM'
                    gdf_bank.at[index,'type'] = brand_name
                    gdf_bank.at[index, 'score'] = score
                    gdf_bank.at[index, 'log'] = f'✅ | Contain ATM | Wrong Subclass | Match with {brand_name}'

                else:
                    gdf_bank.at[index, 'category'] = 'other'
                    gdf_bank.at[index, 'score'] = score
                    gdf_bank.at[index, 'log'] = '❌ | Contain ATM | Wrong Subclass | Score < 75'
        
        elif 'sampah' in name or name.startswith(tuple(COP_KEY)) or any(noise in name for noise in BANKING_NOISE):
            if any(potential in name for potential in BANKING_ATM_KEY):
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'log'] = '❌ | Contain Noise | ATM Match'
            elif any(potential in sub for potential in BANKING_SUB):
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'log'] = '❌ | Contain Noise | Subclass Match'
            elif any(potential in name for potential in BANKING_BANK_BI):
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'log'] = '❌ | Contain Noise | Bank Indonesia'
            elif score > 90:
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'log'] = '❌ | Contain Noise | Brand Match'
            else:
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'log'] = '❌ | Contain Noise | Else'

        elif name.startswith(tuple(BANKING_BANK_BI)):
            if any(potential in sub for potential in BANKING_SUB):
                gdf_bank.at[index, 'category'] = 'Banking'
                gdf_bank.at[index, 'sub_category'] = 'Bank'
                gdf_bank.at[index, 'type'] = 'Bank Indonesia'
                gdf_bank.at[index, 'score'] = 100
                gdf_bank.at[index,'log'] = '✅ | Bank Indonesia | Match Subclass'
                
            else:
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'score'] = score
                gdf_bank.at[index,' log'] = '❌ | Bank Indonesia | Wrong Subclass'
        
        elif 'mandiri' in name and any(potential in name for potential in BANKING_POTENTIAL):
            if any(potential in name for potential in BANKING_SUB): 
                gdf_bank.at[index, 'category'] = 'Banking'
                gdf_bank.at[index, 'sub_category'] = 'Bank'
                gdf_bank.at[index, 'type'] = 'Mandiri'
                gdf_bank.at[index, 'score'] = 100
                gdf_bank.at[index, 'log'] = '✅ | Contain Mandiri | Match Name'

            elif any(potential in sub for potential in BANKING_SUB):
                gdf_bank.at[index, 'category'] = 'Banking'
                gdf_bank.at[index, 'sub_category'] = 'Bank'
                gdf_bank.at[index, 'type'] = 'Mandiri'
                gdf_bank.at[index, 'score'] = 100
                gdf_bank.at[index, 'log'] = '✅ | Contain Mandiri | Match Subclass'

            else:
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'log'] = '❌ | Contain Mandiri | Wrong Subclass'

        else:
            if any(potential in sub for potential in BANKING_SUB):
                if score > 90:
                    gdf_bank.at[index, 'category'] = 'Banking'
                    gdf_bank.at[index, 'sub_category'] = 'Bank'
                    gdf_bank.at[index, 'type'] = brand_name
                    gdf_bank.at[index, 'score'] = score
                    gdf_bank.at[index, 'log'] = '✅| Else | Match Subclass | Score > 90'

                else:
                    if score >= 75:
                        gdf_bank.at[index, 'category'] = 'Banking'
                        gdf_bank.at[index, 'sub_category'] = 'Bank'
                        gdf_bank.at[index, 'type'] = 'other'
                        gdf_bank.at[index, 'score'] = score
                        gdf_bank.at[index, 'log'] = '✅| Else | Match Subclass | Score < 90 | Score > 75'
                    
                    elif name.startswith(tuple(BANKING_POTENTIAL)) and any(
                        potential in sub for potential in BANKING_SUB):
                        gdf_bank.at[index, 'category'] = 'Banking'
                        gdf_bank.at[index, 'sub_category'] = 'Bank'
                        gdf_bank.at[index, 'type'] = 'other'
                        gdf_bank.at[index, 'score'] = score
                        gdf_bank.at[index, 'log'] = '✅| Else | Match Subclass | Score < 90 | Start With Potential'
                    
                    elif name.startswith(tuple(BANKING_POTENTIAL)) and not any(
                        potential in sub for potential in BANKING_SUB):
                        gdf_bank.at[index, 'category'] = 'Banking'
                        gdf_bank.at[index, 'sub_category'] = 'Bank'
                        gdf_bank.at[index, 'type'] = 'other'
                        gdf_bank.at[index, 'score'] = score
                        gdf_bank.at[index, 'log'] = '⚠️| Else | Wrong Subclass | Score < 90 | Start With Potential'

                    elif any(potential in name for potential in BANKING_BANK_START):
                        gdf_bank.at[index, 'category'] = 'Banking'
                        gdf_bank.at[index, 'sub_category'] = 'Bank'
                        gdf_bank.at[index, 'type'] = 'other'
                        gdf_bank.at[index, 'score'] = score
                        gdf_bank.at[index, 'log'] = '✅| Else | Match Subclass | Score < 90 | Local Bank Indicate'
                    
                    else:
                        gdf_bank.at[index, 'category'] = 'Banking'
                        gdf_bank.at[index, 'sub_category'] = 'Agent and Microservices'
                        gdf_bank.at[index, 'type'] = 'other'
                        gdf_bank.at[index, 'score'] = score
                        gdf_bank.at[index, 'log'] = '✅| Else | Agent and Microservices'

            else:
                gdf_bank.at[index, 'category'] = 'other'
                gdf_bank.at[index, 'score'] = score
                gdf_bank.at[index, 'log'] = f'❌ | Else'

    gdf_bank_other = gdf_bank[gdf_bank['category'] == 'other']
    gdf_bank_main = gdf_bank[gdf_bank['sub_category'] == 'Bank']
    gdf_bank_atm = gdf_bank[gdf_bank['sub_category'] == 'ATM']
    gdf_bank_micro = gdf_bank[gdf_bank['sub_category'] == 'Agent and Microservices']

    gdf_bank_main = advance_geodeduplicate(
        gdf_bank_main, 'cleaned_name', 'review', eps=5, sim_treshold=90, min_samples=1)
    gdf_bank_atm = advance_geodeduplicate(
        gdf_bank_atm, 'cleaned_name', 'review', eps=5, sim_treshold=90, min_samples=1)
    
    compile_ = pd.concat([gdf_bank_main, gdf_bank_atm, gdf_bank_micro, gdf_bank_other], ignore_index=True)
    gdf_bank = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs) 

    if verbose: 
        master_list = [
            ['atm', gdf_bank_atm, len(gdf_bank_atm)], 
            ['bank', gdf_bank_main, len(gdf_bank_main)],
            ['micro', gdf_bank_micro, len(gdf_bank_micro)]]
        
        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((atm, bank, micro))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((atm, bank, micro))
        
        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result Is Empty!')
        print(f'Total ATM POI Classified : {result_summary['atm']}')
        print(f'Total Bank POI Classified : {result_summary['bank']}')
        print(f'Total Agent and Microservices POI Classified : {result_summary['micro']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ Banking POI Filtering Finished!')
    
    if verbose:
        if result_gdf is not None: 
            return result_gdf.reset_index(drop=True)
        else: return None
    else: 
        print(f'\n✅ Banking POI Filtering Finished!')
        return gdf_bank.reset_index(drop=True)

def poi_finance(
    gdf_finance, 
    cleaned_name_column, 
    subclass_column,
    verbose = True,
    life = True,
    cop = True,
    pawn = True
    ):

    print('⚙️ Processing Financial Service Category Point of Interest (POI) Filter Object')
    gdf_finance[cleaned_name_column] = gdf_finance[cleaned_name_column].apply(clean_name)
    gdf_finance[subclass_column] = gdf_finance[subclass_column].apply(clean_name)
    gdf_ = len(gdf_finance)

    life, cop, pawn = validate_subclass({'life' : life, 'cop' : cop, 'pawn' : pawn}).values()

    for index, row in tqdm(gdf_finance.iterrows(), total=len(gdf_finance)):
        name = row[cleaned_name_column]
        sub = row[subclass_column]

        if any(potential in sub for potential in FIN_SUB):
            if any(potential in name for potential in LIFE_KEY):
                gdf_finance.at[index, 'category'] = 'Financial Service'
                gdf_finance.at[index, 'sub_category'] = 'Insurance'
                gdf_finance.at[index, 'type'] = None
                gdf_finance.at[index, 'score'] = 100
                gdf_finance.at[index, 'log'] = '✅ | Ins | by Name | Match Name | Match Subclass'
            
            elif not any(potential in name for potential in LIFE_KEY) and 'asuransi' in sub:
                gdf_finance.at[index, 'category'] = 'Financial Service'
                gdf_finance.at[index, 'sub_category'] = 'Insurance'
                gdf_finance.at[index, 'type'] = None
                gdf_finance.at[index, 'score'] = 100
                gdf_finance.at[index, 'log'] = '✅ | Ins | by Name | Match Name | Match Subclass '
            
            elif name.startswith(tuple(COP_KEY)):
                gdf_finance.at[index, 'category'] = 'Financial Service'
                gdf_finance.at[index, 'sub_category'] = 'Cooperation'
                gdf_finance.at[index, 'type'] = None
                gdf_finance.at[index, 'score'] = 100
                gdf_finance.at[index, 'log'] = '✅ | Wrong Subclass | Start With Cop Name'
            
            elif any(potential in name for potential in PAWN_KEY):
                gdf_finance.at[index, 'category'] = 'Financial Service'
                gdf_finance.at[index, 'sub_category'] = 'Pawn Shop'
                gdf_finance.at[index, 'type'] = None
                gdf_finance.at[index, 'score'] = 100
                gdf_finance.at[index, 'log'] = '✅ | Pawn | by Name | Match Name | Match Subclass'
            
            else:
                gdf_finance.at[index, 'category'] = 'other'
                gdf_finance.at[index, 'log'] = '❌ | Wrong Name | Match Subclass'
        
        else:    
            if name.startswith(tuple(PAWN_KEY)):
                gdf_finance.at[index, 'category'] = 'Financial Service'
                gdf_finance.at[index, 'sub_category'] = 'Pawn Shop'
                gdf_finance.at[index, 'type'] = None
                gdf_finance.at[index, 'score'] = 100
                gdf_finance.at[index, 'log'] = '✅ | Wrong Subclass | Start With Pawn Name'
            
            elif name.startswith(tuple(COP_KEY)):
                gdf_finance.at[index, 'category'] = 'Financial Service'
                gdf_finance.at[index, 'sub_category'] = 'Cooperation'
                gdf_finance.at[index, 'type'] = None
                gdf_finance.at[index, 'score'] = 100
                gdf_finance.at[index, 'log'] = '✅ | Wrong Subclass | Start With Cop Name'
                
            elif any(potential in name for potential in LIFE_KEY):
                gdf_finance.at[index, 'category'] = 'other'
                gdf_finance.at[index, 'log'] = '❌ | Ins | Match Name | Wrong Subclass'

            elif any(potential in name for potential in COP_KEY):
                gdf_finance.at[index, 'category'] = 'other'
                gdf_finance.at[index, 'log'] = '❌ | Cop | Match Name | Wrong Subclass'

            elif any(potential in name for potential in PAWN_KEY):
                gdf_finance.at[index, 'category'] = 'other'
                gdf_finance.at[index, 'log'] = '❌ | Pawn | Match Name | Wrong Subclass'
            
            else:
                gdf_finance.at[index, 'category'] = 'other'
                gdf_finance.at[index, 'log'] = '❌ | Else '
    
    gdf_finance_other = gdf_finance[gdf_finance['category'] == 'other']
    gdf_life = gdf_finance[gdf_finance['sub_category'] == 'Insurance']
    gdf_cop = gdf_finance[gdf_finance['sub_category'] == 'Cooperation']
    gdf_pawn = gdf_finance[gdf_finance['sub_category'] == 'Pawn Shop']

    gdf_life = advance_geodeduplicate(gdf_life, 'cleaned_name', 'review', eps=10, sim_treshold=95)
    gdf_cop = advance_geodeduplicate(gdf_cop, 'cleaned_name', 'review', eps=10, sim_treshold=95)
    gdf_pawn = advance_geodeduplicate(gdf_pawn, 'cleaned_name', 'review', eps=10, sim_treshold=95)
    
    compile_ = pd.concat([gdf_life, gdf_cop, gdf_pawn, gdf_finance_other], ignore_index=True)
    gdf_finance = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs) 

    if verbose: 
        master_list = [
            ['life', gdf_life, len(gdf_life)], 
            ['cop', gdf_cop, len(gdf_cop)],
            ['pawn', gdf_pawn, len(gdf_pawn)]] 
        
        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((life, cop, pawn))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((life, cop, pawn))
        
        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result Is Empty!')
        print(f'Total Insurance POI Classified : {result_summary['life']}')
        print(f'Total Cooperation POI Classified : {result_summary['cop']}')
        print(f'Total Pawn Shop POI Classified : {result_summary['pawn']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ Financial Service POI Filtering Finished!')
    
    if verbose:
        if result_gdf is not None: 
            return result_gdf.reset_index(drop=True)
        else: return None
    else: 
        print(f'\n✅ Financial Service POI Filtering Finished!')
        return gdf_finance.reset_index(drop=True)

def poi_health(
    gdf_health,
    cleaned_name_column,
    subclass_column,
    review_column,
    verbose = True,
    hospital = True,
    clinic = True,
    pharmacy = True
    ):

    print('⚙️ Processing Health Category Point of Interest (POI) Filter Object')
    gdf_health[cleaned_name_column] = gdf_health[cleaned_name_column].apply(clean_name)
    gdf_health[subclass_column] = gdf_health[subclass_column].apply(clean_name)
    gdf_ = len(gdf_health)

    hospital, clinic, pharmacy = validate_subclass(
        {'hospital': hospital, 'clinic' : clinic, 'pharmacy': pharmacy}).values()

    for index, row in tqdm(gdf_health.iterrows(), total = len(gdf_health)): 
        name = row[cleaned_name_column]
        sub = row[subclass_column]
        review = row[review_column]

        if name.startswith(tuple(HEALTH_HOSPITAL_KEY_1)):
            if not any(potential in sub for potential in HEALTH_SUB):
                if review > 10:
                    gdf_health.at[index, 'category'] = 'Healthcare'
                    gdf_health.at[index, 'sub_category'] = 'Hospital'
                    gdf_health.at[index, 'type'] = None
                    gdf_health.at[index, 'score'] = 100
                    gdf_health.at[index, 'log'] = '✅ | Argue Subclass High Review | Start with Hospital Potential'
                    continue

                gdf_health.at[index, 'category'] = 'other' 
                gdf_health.at[index, 'log'] = '❌ | Wrong Subclass! | Start With Hospital Potential'
                continue

            gdf_health.at[index, 'category'] = 'Healthcare'
            gdf_health.at[index, 'sub_category'] = 'Hospital'
            gdf_health.at[index, 'type'] = None
            gdf_health.at[index, 'score'] = 100
            gdf_health.at[index, 'log'] = '✅ | Match Subclass | Start with Hospital Potential'

        elif any(potential in name for potential in HEALTH_HOSPITAL_KEY_2):
            if not any(potential in sub for potential in HEALTH_SUB):
                gdf_health.at[index, 'category'] = 'other'
                gdf_health.at[index, 'log'] = '❌ | Wrong Subclass! | Contain Hospital'
                continue

            elif any(noise in name for noise in HEALTH_NOISE):
                gdf_health.at[index, 'category'] = 'other'
                gdf_health.at[index, 'log'] = '❌  | Match Subclass | Contain Noise | Contain Hospital'
                continue

            else:
                gdf_health.at[index, 'category'] = 'Healthcare'
                gdf_health.at[index, 'sub_category'] = 'Hospital'
                gdf_health.at[index, 'type'] = None
                gdf_health.at[index, 'score'] = 100
                gdf_health.at[index, 'log'] = '✅ | Match Subclass | Contain Hospital'

        elif any(potential in sub for potential in HEALTH_PHARMACY_SUB) and any(
            potential in name for potential in HEALTH_PHARMACY_BRAND):
            gdf_health.at[index, 'category'] = 'Healthcare'
            gdf_health.at[index, 'sub_category'] = 'Pharmacy'
            gdf_health.at[index, 'type'] = None
            gdf_health.at[index, 'score'] = 100
            gdf_health.at[index, 'log'] = ' ✅ | Match Subclass | Match Brand Pharmacy'

        elif any(potential in sub for potential in HEALTH_PHARMACY_SUB) and any(
            potential in name for potential in HEALTH_PHARMACY_KEY):
            gdf_health.at[index, 'category'] = 'Healthcare'
            gdf_health.at[index, 'sub_category'] = 'Pharmacy'
            gdf_health.at[index, 'type'] = None
            gdf_health.at[index, 'score'] = 100
            gdf_health.at[index, 'log'] = '✅ | Match Subclass | Match Name Pharmacy'
        
        elif name.startswith(tuple(HEALTH_PHARMACY_KEY)):
            gdf_health.at[index, 'category'] = 'Healthcare'
            gdf_health.at[index, 'sub_category'] = 'Pharmacy'
            gdf_health.at[index, 'type'] = None
            gdf_health.at[index, 'score'] = 100
            gdf_health.at[index, 'log'] = '✅ | Match Subclass | Match Name Pharmacy'

        elif any(potential in name for potential in HEALTH_CLINIC_KEY):
            if not any(potential in sub for potential in HEALTH_SUB) or 'fakultas' in name:
                gdf_health.at[index, 'category'] = 'other'
                gdf_health.at[index, 'log'] = '❌ | Wrong Subclass! | Contain Clinic Potential'
                continue

            gdf_health.at[index, 'category'] = 'Healthcare'
            gdf_health.at[index, 'sub_category'] = 'Clinic'
            gdf_health.at[index, 'type'] = None
            gdf_health.at[index, 'score'] = 100
            gdf_health.at[index, 'log'] = '✅ | Match Subclass | Contain Clinic potential'

        else:
            gdf_health.at[index, 'category'] = 'other' 
            gdf_health.at[index, 'log'] = '❌ | Else'
    
        if any(stopword in name for stopword in HEALTH_HOSPITAL_STOPWORD):
            gdf_health.at[index, 'temp_name'] = remove_stopword(name, HEALTH_HOSPITAL_STOPWORD)
        else:
            gdf_health.at[index, 'temp_name'] = name
    
    gdf_health_other = gdf_health[gdf_health['category'] == 'other']
    gdf_hospital = gdf_health[gdf_health['sub_category'] == 'Hospital']
    gdf_clinic = gdf_health[gdf_health['sub_category'] == 'Clinic']
    gdf_pharmacy = gdf_health[gdf_health['sub_category'] == 'Pharmacy']

    gdf_pharmacy = advance_geodeduplicate(
        gdf_pharmacy, 'cleaned_name', 'review', eps=10, sim_treshold=85, min_samples=1)
    
    gdf_clinic = advance_geodeduplicate(
        gdf_clinic, 'temp_name', 'review', eps=10, sim_treshold=85, min_samples=1)
    
    gdf_hospital = advance_geodeduplicate(
        gdf_hospital, 'temp_name', 'review', eps=300, sim_treshold=85, min_samples=1)

    compile_ = pd.concat([gdf_pharmacy, gdf_clinic, gdf_hospital, gdf_health_other], ignore_index=True)
    gdf_health = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs)

    if verbose:
        master_list = [
            ['hospital', gdf_hospital, len(gdf_hospital)],
            ['clinic', gdf_clinic, len(gdf_clinic), gdf_clinic], 
            ['pharmacy', gdf_pharmacy ,len(gdf_pharmacy)]
        ]
        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((hospital, clinic, pharmacy))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((hospital, clinic, pharmacy))

        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result is Empty!')
        print(f'Total Hospital POI Classified : {result_summary['hospital']}')
        print(f'Total Clinic POI Classified : {result_summary['clinic']}')
        print(f'Total Pharmacy POI Classified : {result_summary['pharmacy']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ Health POI Filtering Finished!')

    if verbose:
        if result_gdf is not None: 
            return result_gdf.reset_index(drop=True)
        else: return None
    else:
        print(f'\n✅ Health POI Filtering Finished!')
        return gdf_health.reset_index(drop=True)
        
def poi_school(
    gdf_school,
    cleaned_name_column,
    subclass_column,
    verbose = True,
    pre_school = True,
    elementary_school = True,
    junior_high_school = True,
    senior_high_school = True,
    international = True,
    special = True,
    non_formal = True,
    unit = True
    ):
    
    print('⚙️ Processing School Category Point of Interest (POI) Filter Object')
    gdf_school[cleaned_name_column] = gdf_school[cleaned_name_column].apply(clean_name)
    gdf_school[subclass_column] = gdf_school[subclass_column].apply(clean_name)
    gdf_school = set_value(gdf_school)
    gdf_ = len(gdf_school)

    pre_school, elementary_school, junior_high_school, senior_high_school, international, special, non_formal, unit = validate_subclass({
        'pre_school' : pre_school,
        'elementary_school' : elementary_school,  
        'junior_high_school' : junior_high_school, 
        'senior_high_school' : senior_high_school,
        'international' : international,
        'special' : special,
        'non_formal' : non_formal,
        'unit' : unit
    }).values()
    
    for index, row in tqdm(gdf_school.iterrows(), total = len(gdf_school)):
        name = str(row[cleaned_name_column]).lower()
        sub = str(row[subclass_column]).lower().strip()

        if not any(potential in sub for potential in SCHOOL_SUB):
            gdf_school.at[index, 'category'] = 'other'
            gdf_school.at[index, 'log'] = '❌ | Wrong Subclass!'
            continue
        
        if name.startswith(tuple(SCHOOL_SD_KEY)) or name in SCHOOL_SD_KEY:
            gdf_school.at[index, 'category'] = 'School'
            gdf_school.at[index, 'sub_category'] = 'Elementary School'
            gdf_school.at[index, 'type'] = None
            gdf_school.at[index, 'score'] = 100
            gdf_school.at[index, 'log'] = '✅ | No Noise | Start with Elementary School Word List'
        
        elif name.startswith(tuple(SCHOOL_TK_KEY)) or name in SCHOOL_TK_KEY:
            gdf_school.at[index, 'category'] = 'School'
            gdf_school.at[index, 'sub_category'] = 'Pre-school'
            gdf_school.at[index, 'type'] = None
            gdf_school.at[index, 'score'] = 100
            gdf_school.at[index, 'log'] = '✅ | No Noise | Start with Pre-school Word List'

        elif name.startswith(tuple(SCHOOL_SMP_KEY)) or name in SCHOOL_SMP_KEY:
            gdf_school.at[index, 'category'] = 'School'
            gdf_school.at[index, 'sub_category'] = 'Junior High School'
            gdf_school.at[index, 'type'] = None
            gdf_school.at[index, 'score'] = 100
            gdf_school.at[index, 'log'] = '✅ | No Noise | Start with Junior High School Word List'
        
        elif name.startswith(tuple(SCHOOL_SMA_KEY)) or name in SCHOOL_SMA_KEY:
            gdf_school.at[index, 'category'] = 'School'
            gdf_school.at[index, 'sub_category'] = 'Senior High School'
            gdf_school.at[index, 'type'] = None
            gdf_school.at[index, 'score'] = 100
            gdf_school.at[index, 'log'] = '✅ | No Noise | Start with Senior High School Word List'
        
        elif name.startswith(tuple(SCHOOL_INTER_KEY)):
            if all(word not in sub for word in SCHOOL_SUB):
                gdf_school.at[index, 'category'] = 'other'
                gdf_school.at[index, 'log'] = '❌ | Wrong Subclass! | Start with International School Word List'
                
            else:
                gdf_school.at[index, 'category'] = 'School'
                gdf_school.at[index, 'sub_category'] = 'International School'
                gdf_school.at[index, 'type'] = 'Private'
                gdf_school.at[index, 'score'] = 100
                gdf_school.at[index, 'log'] = '✅ | No Noise | Start with International School Word List'
                

        elif any(potential in name for potential in SCHOOL_INTER_KEY):
            if all(word not in sub for word in SCHOOL_SUB):
                gdf_school.at[index, 'category'] = 'other'
                gdf_school.at[index, 'log'] = '❌ | Wrong Subclass! | Contain International School Word List'
            
            else:
                gdf_school.at[index, 'category'] = 'School'
                gdf_school.at[index, 'sub_category'] = 'International School'
                gdf_school.at[index, 'type'] = 'Private'
                gdf_school.at[index, 'score'] = 100
                gdf_school.at[index, 'log'] = '✅ | No Noise | Contain International School Word List'
        
        elif any(potential in name for potential in SCHOOL_NONFORMAL_KEY):
            if all(word not in sub for word in SCHOOL_SUB):
                gdf_school.at[index, 'category'] = 'other'
                gdf_school.at[index, 'log'] = '❌ | Wrong Subclass! | Contain Non-Formal Education Word List'
            
            else:
                gdf_school.at[index, 'category'] = 'School'
                gdf_school.at[index, 'sub_category'] = 'Non-Formal Education'
                gdf_school.at[index, 'type'] = None
                gdf_school.at[index, 'score'] = 100
                gdf_school.at[index, 'log'] = '✅ | No Noise | Contain Non-Formal Education Word List'
        
        elif any(potential in name for potential in SCHOOL_UNIT_KEY):
            if all(word not in sub for word in SCHOOL_SUB):
                gdf_school.at[index, 'category'] = 'other'
                gdf_school.at[index, 'log'] = '❌ | Wrong Subclass! | Contain Unit School Word List'
            
            else:
                gdf_school.at[index, 'category'] = 'School'
                gdf_school.at[index, 'sub_category'] = 'Techincal Implementation Unit'
                gdf_school.at[index, 'type'] = None
                gdf_school.at[index, 'score'] = 100
                gdf_school.at[index, 'log'] = '✅ | No Noise | Contain Unit School Word List'
        
        elif any(potential in name for potential in SCHOOL_SPECIAL_KEY):
            if all(word not in sub for word in SCHOOL_SUB):
                gdf_school.at[index, 'category'] = 'other'
                gdf_school.at[index, 'log'] = '❌ | Wrong Subclass! | Contain Special School Word List'
            
            else:
                gdf_school.at[index, 'category'] = 'School'
                gdf_school.at[index, 'sub_category'] = 'Special Education School'
                gdf_school.at[index, 'type'] = None
                gdf_school.at[index, 'score'] = 100
                gdf_school.at[index, 'log'] = '✅ | No Noise | Contain Special School Word List'

        else:
            if all(word not in sub for word in SCHOOL_SUB):
                gdf_school.at[index, 'category'] = 'other'
                gdf_school.at[index, 'log'] = '❌ | Wrong Subclass! | No Noise | Else'

            else:
                gdf_school.at[index, 'category'] = 'other'
                gdf_school.at[index, 'log'] = '❌ | No Noise | Else'

        best_match, score, brand = match_brand(name, SCHOOL_SWASTA_BRAND, threshold=95)
        if score > 95:
            gdf_school.at[index, 'type'] = brand
        elif any(potential in name for potential in SCHOOL_NEGERI_KEY):
            gdf_school.at[index, 'type'] = 'Public'
        elif any(potential in name for potential in SCHOOL_SWASTA_KEY):
            gdf_school.at[index, 'type'] = 'Private'
        else:
            gdf_school.at[index, 'type'] = 'Public' 
        
        # if all(word not in sub for word in SCHOOL_SUB):
        #     gdf_school.at[index, 'category'] = 'other'
        #     gdf_school.at[index, 'log'] = '❌ | Wrong Subclass!'
        #     continue
    
    gdf_school_other = gdf_school[gdf_school['category'] == 'other']
    gdf_pre = gdf_school[gdf_school['sub_category'] == 'Pre-school']
    gdf_element = gdf_school[gdf_school['sub_category'] == 'Elementary School']
    gdf_junior = gdf_school[gdf_school['sub_category'] == 'Junior High School']
    gdf_senior = gdf_school[gdf_school['sub_category'] == 'Senior High School']
    gdf_inter = gdf_school[gdf_school['sub_category'] == 'International School']
    gdf_special = gdf_school[gdf_school['sub_category'] == 'Special Education School']
    gdf_nonformal = gdf_school[gdf_school['sub_category'] == 'Non-Formal Education']
    gdf_unit = gdf_school[gdf_school['sub_category'] == 'Techincal Implementation Unit']

    compile_ = pd.concat(
        [gdf_school_other, gdf_pre, gdf_element, gdf_junior, gdf_senior, gdf_inter, gdf_special, gdf_nonformal, gdf_unit],
        ignore_index= True,)
    gdf_school = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs)

    if verbose:
        master_list = [
            ['pre_school', gdf_pre, len(gdf_pre)],
            ['elementary_school', gdf_element, len(gdf_element)],
            ['junior_high_school', gdf_junior, len(gdf_junior)],
            ['senior_high_school', gdf_senior, len(gdf_senior)],
            ['international', gdf_inter, len(gdf_inter)],
            ['non_formal', gdf_nonformal, len(gdf_nonformal)],
            ['unit', gdf_unit, len(gdf_unit)],
            ['special', gdf_special, len(gdf_special)]
        ]

        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((
            pre_school, elementary_school, junior_high_school, senior_high_school, international, special, non_formal, unit
        ))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((
            pre_school, elementary_school, junior_high_school, senior_high_school, international, special, non_formal, unit
        ))

        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result is Empty!')
        print(f'Total Pre-School POI Classified : {result_summary['pre_school']}')
        print(f'Total Elementary School POI Classified : {result_summary['elementary_school']}')
        print(f'Total Junior High School POI Classified : {result_summary['junior_high_school']}')
        print(f'Total Senior high School POI Classified : {result_summary['senior_high_school']}')
        print(f'Total International School POI Classified : {result_summary['international']}')
        print(f'Total Special Education School POI Classified : {result_summary['special']}')
        print(f'Total Non-Formal Education POI Classified : {result_summary['non_formal']}')
        print(f'Total Technical Implementation Unit POI Classified : {result_summary['unit']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ School POI Filtering Finished!')

    if verbose:
        if result_gdf is not None: 
            return result_gdf.reset_index(drop=True)
    else: 
        return gdf_school.reset_index(drop=True)

def poi_shopping(
    gdf_store,
    cleaned_name_column,
    subclass_column,
    review_column,
    review_threshold=300,
    verbose = True,
    small = True,
    convenience = True,
    supermarket = True,
    mall = True
    ):

    print('⚙️ Processing Retail & Shopping Category Point of Interest (POI) Filter Object')
    gdf_store[cleaned_name_column] = gdf_store[cleaned_name_column].apply(clean_name)
    gdf_store[subclass_column] = gdf_store[subclass_column].apply(clean_name)
    gdf_ = len(gdf_store)

    small, convenience, supermarket, mall = validate_subclass({
        'small': mall, 'convenience' : convenience, 'supermarket' : supermarket, 'mall' : mall
    }).values()

    for index, row in tqdm(gdf_store.iterrows(), total = len(gdf_store)):
        name = row[cleaned_name_column]
        review = int(row[review_column])
        sub = row[subclass_column]

        if not name.startswith(tuple(STORE_START_NOISE)) and any(
            potential in sub for potential in STORE_SUB):

            if any(potential in name for potential in STORE_MINIMARKET_KEY):
                best_match_mm, score_mm = correct_typo_token(name, STORE_MINIMARKET_KEY, threshold=95)
                if score_mm > 95: gdf_store.at[index, 'type'] = best_match_mm.capitalize()
                else: gdf_store.at[index, 'type'] = None

                gdf_store.at[index, 'category'] = 'Retail & Shopping'
                gdf_store.at[index, 'sub_category'] = 'Convenience Store'
                gdf_store.at[index, 'score'] = 100
                gdf_store.at[index, 'log'] = '✅ | Contain Convenience store Potential'
            
            elif review > review_threshold:
                if any(potential in name for potential in STORE_SUPERMARKET_KEY):
                    best_match_sm, score_sm = correct_typo_token(name, STORE_SUPERMARKET_KEY, threshold=95)

                    if score_sm > 95: gdf_store.at[index, 'type'] = best_match_sm.capitalize()
                    else: gdf_store.at[index, 'type'] = None

                    gdf_store.at[index, 'category'] = 'Retail & Shopping'
                    gdf_store.at[index, 'sub_category'] = 'Super Market'
                    gdf_store.at[index, 'type'] = best_match_sm.capitalize()
                    gdf_store.at[index, 'score'] = score_sm
                    gdf_store.at[index, 'log'] = '✅ | Contain Super Market Poential'
                
                elif any(potential in name for potential in STORE_MALL_KEY):
                    best_match_m, score_m = correct_typo_token(name, STORE_MALL_KEY, threshold=95)

                    if score_m > 95: gdf_store.at[index, 'type'] = best_match_m.capitalize()
                    else: gdf_store.at[index, 'type'] = None

                    gdf_store.at[index, 'category'] = 'Retail & Shopping'
                    gdf_store.at[index, 'sub_category'] = 'Mall'
                    gdf_store.at[index, 'type'] = best_match_m.capitalize()
                    gdf_store.at[index, 'score'] = 100
                    gdf_store.at[index, 'log'] = '✅ | Contain Mall Poential'

                else: 
                    if review > 5000:
                        gdf_store.at[index, 'category'] = 'Retail & Shopping'
                        gdf_store.at[index, 'sub_category'] = 'Mall'
                        gdf_store.at[index, 'type'] = None
                        gdf_store.at[index, 'score'] = 100
                        gdf_store.at[index, 'log'] = '✅ | High Review | > 5000 | Mall Indicate'

                    else:
                        gdf_store.at[index, 'category'] = 'Retail & Shopping'
                        gdf_store.at[index, 'sub_category'] = 'Mall'
                        gdf_store.at[index, 'type'] = None
                        gdf_store.at[index, 'score'] = 100
                        gdf_store.at[index, 'log'] = '✅ | High Review | < 5000 | Supermarket Indicate'

            elif any(potential in name for potential in STORE_SUPERMARKET_KEY) and not any(
                noise in name for noise in STORE_NOISE):
                best_match_sm, score_sm = correct_typo_token(name, STORE_SUPERMARKET_KEY, threshold=95)
        
                if score_sm > 95: gdf_store.at[index, 'type'] = best_match_sm.capitalize()
                else: gdf_store.at[index, 'type'] = None

                gdf_store.at[index, 'category'] = 'Retail & Shopping'
                gdf_store.at[index, 'sub_category'] = 'Super Market'
                gdf_store.at[index, 'score'] = 100
                gdf_store.at[index, 'log'] = '✅ | Contain Supermarket Potential'
            
            else:
                gdf_store.at[index, 'category'] = 'Retail & Shopping'
                gdf_store.at[index, 'sub_category'] = 'Small Store'
                gdf_store.at[index, 'type'] = None
                gdf_store.at[index, 'score'] = 100
                gdf_store.at[index, 'log'] = '✅ | Match Subclass | Small Store Indicate'

        elif any(potential in sub for potential in STORE_SUB):
            gdf_store.at[index, 'category'] = 'Retail & Shopping'
            gdf_store.at[index, 'sub_category'] = 'Small Store'
            gdf_store.at[index, 'type'] = None
            gdf_store.at[index, 'score'] = 100
            gdf_store.at[index, 'log'] = '✅ | Contain | Small Store Indicate'
        
        elif name.startswith(tuple(STORE_NOISE)):
            gdf_store.at[index, 'category'] = 'Retail & Shopping'
            gdf_store.at[index, 'sub_category'] = 'Small Store'
            gdf_store.at[index, 'type'] = None
            gdf_store.at[index, 'score'] = 100
            gdf_store.at[index, 'log'] = '✅ | Start | Small Store Indicate'

        else:
            gdf_store.at[index, 'category'] = 'other'
            gdf_store.at[index, 'log'] = '❌ | Else'

    gdf_store_other = gdf_store[gdf_store['category'] == 'other']
    gdf_small = gdf_store[gdf_store['sub_category'] == 'Small Store']
    gdf_convenience = gdf_store[gdf_store['sub_category'] == 'Convenience Store']
    gdf_supermarket = gdf_store[gdf_store['sub_category'] == 'Super Market']
    gdf_mall = gdf_store[gdf_store['sub_category'] == 'Mall']

    gdf_mall = geodeduplicate(gdf_mall, eps=100)
    gdf_mall = advance_geodeduplicate(gdf_mall, 'cleaned_name', 'review', eps=200, sim_treshold=85)

    compile = pd.concat([gdf_mall, gdf_supermarket,], ignore_index=True)
    gdf_compile = gpd.GeoDataFrame(compile, geometry='geometry', crs=compile.crs)

    gdf_compile = geodeduplicate(gdf_compile, eps=50)
    gdf_compile = advance_geodeduplicate(gdf_compile, 'cleaned_name', 'review', eps=250, sim_treshold=75)

    final_compile = pd.concat([gdf_compile, gdf_convenience], ignore_index=True)
    gdf_store = gpd.GeoDataFrame(final_compile, geometry='geometry', crs=final_compile.crs)

    gdf_convenience = gdf_store[gdf_store['sub_category'] == 'Convenience Store']
    gdf_supermarket = gdf_store[gdf_store['sub_category'] == 'Super Market']
    gdf_mall = gdf_store[gdf_store['sub_category'] == 'Mall']

    compile_ = pd.concat([gdf_small, gdf_convenience, gdf_supermarket, gdf_mall, gdf_store_other],ignore_index=True)
    gdf_store = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs)

    if verbose:
        master_list = [
            ['small', gdf_small, len(gdf_small)],
            ['convenience', gdf_convenience, len(gdf_convenience)],
            ['supermarket', gdf_supermarket, len(gdf_supermarket)],
            ['mall', gdf_mall, len(gdf_mall)]
        ]
        
        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((small, convenience, supermarket, mall))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((small, convenience, supermarket, mall))

        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result is Empty!')
        print(f'Total Small Store POI Classified : {result_summary['small']}')
        print(f'Total Mini Market POI Classified : {result_summary['convenience']}')
        print(f'Total Super Market POI Classified : {result_summary['supermarket']}')
        print(f'Total Mall POI Classified : {result_summary['mall']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ Retail and Shopping POI Filtering Finished!')
    
    if verbose:
        if result_gdf is not None: 
            return result_gdf.reset_index(drop=True)
    else: 
        print(f'\n✅ Retail and Shopping POI Filtering Finished!')
        return gdf_store.reset_index(drop=True)

def poi_univ(gdf_univ, cleaned_name_column, subclass_column, review_column, upper_review=500, verbose=True,
    university = True, institute = True, polytechnic = True, college = True, academy = True):

    print('⚙️ Processing Higher Education Category Point of Interest (POI) Filter Object')
    gdf_univ[cleaned_name_column] = gdf_univ[cleaned_name_column].apply(clean_name)
    gdf_univ[subclass_column] = gdf_univ[subclass_column].apply(clean_name)
    gdf_ = len(gdf_univ)

    university, institute, polytechnic, college, academy = validate_subclass({
        'university' : university, 
        'institute' : institute, 
        'polytechnic' : polytechnic, 
        'college' : college,
        'academy' : academy
    }).values()
    
    for index, row in tqdm(gdf_univ.iterrows(), total = len(gdf_univ)):
        name = str(row[cleaned_name_column]).lower()
        sub = str(row[subclass_column]).lower().strip()
        review = row[review_column]

        best_match, score = correct_typo_dict(name, UNIV_POTENTIAL, threshold=95)
        best_match_type, score_type, brand_type = match_brand(name, UNIV_SWASTA_BRAND, threshold=95)

        if any(noise in name for noise in UNIV_NOISE):
            gdf_univ.at[index, 'category'] = 'other'
            gdf_univ.at[index, 'log'] = '❌ | Contain Noise word'

        elif name == 'universitas indonesia':
            gdf_univ.at[index, 'category'] = 'Higher Education'
            gdf_univ.at[index, 'sub_category'] = 'University'
            gdf_univ.at[index, 'type'] = 'Public'
            gdf_univ.at[index, 'score'] = 100
            gdf_univ.at[index, 'log'] = '✅ | Match Universitas Indonesia'
            continue

        elif name.startswith(tuple(UNIV_ENTRY)):
            if all(potential not in sub for potential in UNIV_SUB):
                gdf_univ.at[index, 'category'] = 'other'
                gdf_univ.at[index, 'log'] = '❌ | Subclass not Match'
                continue

            gdf_univ.at[index, 'category'] = 'Higher Education'
            gdf_univ.at[index, 'sub_category'] = best_match.capitalize()
            gdf_univ.at[index, 'type'] = None
            gdf_univ.at[index, 'score'] = score
            gdf_univ.at[index, 'log'] = f'✅ | Start With | Entry Word'

        elif score > 90:
            if review >= 5:
                if all(potential not in sub for potential in UNIV_SUB):
                    gdf_univ.at[index, 'category'] = 'other'
                    gdf_univ.at[index, 'log'] = '❌ | Subclass not Match'
                    continue
                
                if any(noise in sub for noise in UNIV_SUB_NOISE):
                    gdf_univ.at[index, 'category'] = 'other'
                    gdf_univ.at[index, 'log'] = '❌ | Subclass not Match'
                    continue

                gdf_univ.at[index, 'category'] = 'Higher Education'
                gdf_univ.at[index, 'sub_category'] = best_match.capitalize()
                gdf_univ.at[index, 'type'] = None
                gdf_univ.at[index, 'score'] = score
                gdf_univ.at[index, 'log'] = f' ✅ | Match With | {best_match.capitalize()}'

            else:
                gdf_univ.at[index, 'category'] = 'other'
                gdf_univ.at[index, 'log'] = '❌ | Low Review | Fake University Alert'
                continue

        else:
            gdf_univ.at[index, 'category'] = 'other'
            gdf_univ.at[index, 'log'] = '❌ | Else'
        
        if 'sekolah' in name and not 'tinggi' in name:
            gdf_univ.at[index, 'category'] = 'other'
            gdf_univ.at[index, 'log'] = '❌ | Detected as School POI, not College!'
        
        if score_type > 90:
            gdf_univ.at[index, 'type'] = 'Public'
        else:
            gdf_univ.at[index, 'type'] = 'Private'
    
    gdf_univ_other = gdf_univ[gdf_univ['category'] == 'other']
    gdf_univ = gdf_univ[gdf_univ['category'] != 'other'].copy()
    gdf_univ['cleaned_name'] = gdf_univ['cleaned_name'].apply(remove_stopword, stopword_list=UNIV_STOPWORDS)
    gdf_univ = advance_geodeduplicate(gdf_univ, 'cleaned_name', review_column, eps=500, sim_treshold=85)

    gdf_university = gdf_univ[gdf_univ['sub_category'] == 'University']
    gdf_instiute = gdf_univ[gdf_univ['sub_category'] == 'Institute']
    gdf_polytechnic = gdf_univ[gdf_univ['sub_category'] == 'Polytechnic']
    gdf_college = gdf_univ[gdf_univ['sub_category'] == 'College']
    gdf_academy = gdf_univ[gdf_univ['sub_category'] == 'Academy']

    compile_ = pd.concat([gdf_university, gdf_instiute, gdf_polytechnic, gdf_college, gdf_academy, gdf_univ_other],
        ignore_index=True)
    gdf_univ = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs)

    if verbose:
        master_list = [
            ['university', gdf_university, len(gdf_university)],
            ['institute', gdf_instiute, len(gdf_instiute)],
            ['polytechnic', gdf_polytechnic, len(gdf_polytechnic)],
            ['college', gdf_college, len(gdf_college)],
            ['academy', gdf_academy, len(gdf_academy)]
        ]

        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((university, institute, polytechnic, college, academy))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((university, institute, polytechnic, college, academy))

        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result is Empty!')
        print(f'Total University POI Classified : {result_summary['university']}')
        print(f'Total Institue POI Classified : {result_summary['institute']}')
        print(f'Total Polytechnic POI Classified : {result_summary['polytechnic']}')
        print(f'Total College POI Classified : {result_summary['college']}')
        print(f'Total Academy POI Classified : {result_summary['academy']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ Highed Education POI Filtering Finished!')

    if verbose:
        if result_gdf is not None: return result_gdf.reset_index(drop=True)
        else: return None
    else:
        print(f'\n✅ Highed Education POI Filtering Finished!')
        return gdf_univ
    
def poi_hospitality(
    gdf_hospitality,
    cleaned_name_column,
    subclass_column,
    review_column,
    min_review=5,
    verbose = True, 
    hotel = True,
    villa = True,
    guest = True,
    board = True,
    apart = True
    ):

    print('⚙️ Processing Hospitality & Accomodataion Category Point of Interest (POI) Filter Object')
    gdf_hospitality[cleaned_name_column] = gdf_hospitality[cleaned_name_column].apply(clean_name)
    gdf_hospitality[subclass_column] = gdf_hospitality[subclass_column].apply(clean_name)
    gdf_ = len(gdf_hospitality)

    hotel, villa, guest, board, apart = validate_subclass({
        'hotel' : hotel, 'villa' : villa, 'guest' : guest, 'board' : board, 'apart' : apart
        }).values()
    
    for index, row in tqdm(gdf_hospitality.iterrows(), total = len(gdf_hospitality)):
        name = str(row[cleaned_name_column])
        sub = str(row[subclass_column])
        review = row[review_column]

        if 'apart' in name and any(potential in sub for potential in HOSPITALITY_SUB):
            gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
            gdf_hospitality.at[index, 'sub_category'] = 'Apartement'
            gdf_hospitality.at[index, 'type'] = None
            gdf_hospitality.at[index, 'score'] = 100
            gdf_hospitality.at[index, 'log'] = '✅ | Apartement in Name'
        
        elif 'apart' in sub:
            gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
            gdf_hospitality.at[index, 'sub_category'] = 'Apartement'
            gdf_hospitality.at[index, 'type'] = None
            gdf_hospitality.at[index, 'score'] = 100
            gdf_hospitality.at[index, 'log'] = '✅ | Apartement in Sub'

        elif not any(noise in name for noise in HOSPITALITY_NOISE):
            if any(potential in name for potential in HOSPITALITY_VILLA_KEY):
                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Villa'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Villa in Name'
            
            elif any(potential in name for potential in HOSPITALITY_HOTEL_KEY):
                if not any(potential in sub for potential in HOSPITALITY_SUB):
                    gdf_hospitality.at[index, 'category'] = 'other'
                    gdf_hospitality.at[index, 'log'] = '❌ | Wrong Subclass for Hotel?'
                    continue

                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Hotel'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Hotel/Motel in Name'

            elif any(potential in sub for potential in HOSPITALITY_VILLA_KEY):
                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Villa'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Villa in Sub'
            
            elif any(potential in sub for potential in HOSPITALITY_HOTEL_KEY):
                if 'sekolah' in sub or 'sekolah' in name:
                    gdf_hospitality.at[index, 'category'] = 'other'
                    gdf_hospitality.at[index, 'log'] = '❌ | Contain Sekolah | Hotel/Motel in Sub'
                    continue
            
                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Hotel'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Hotel/Motel in Subclass'
            
            elif any(potential in name for potential in HOSPITALITY_GUEST_KEY):
                if not any(potential in name for potential in HOSPITALITY_SUB):
                    gdf_hospitality.at[index, 'category'] = 'other'
                    gdf_hospitality.at[index, 'log'] = '❌ | Wrong Subclass for Guest House?'
                    continue
                
                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Guest House'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Guest House in Name'  
        
            elif any(potential in sub for potential in HOSPITALITY_GUEST_KEY):
                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Guest House'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Guest House in Sub'

        else:
            if name.startswith(tuple(HOSPITALITY_BOARD_KEY)):
                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Boarding House'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Guest House in Sub'

            elif any(potential in name for potential in HOSPITALITY_BOARD_KEY) and any(
                potential in sub for potential in HOSPITALITY_SUB):
                gdf_hospitality.at[index, 'category'] = 'Hospitality & Accommodation'
                gdf_hospitality.at[index, 'sub_category'] = 'Boarding House'
                gdf_hospitality.at[index, 'type'] = None
                gdf_hospitality.at[index, 'score'] = 100
                gdf_hospitality.at[index, 'log'] = '✅ | Guest House in Sub'
            
            else:
                gdf_hospitality.at[index, 'category'] = 'other'
                gdf_hospitality.at[index, 'log'] = '❌ | Wrong Subclass for Guest House?'
            
        if review > 10000:
            gdf_hospitality.at[index, 'type'] = 'Prioritas Utama'
        elif review > 1000:
            gdf_hospitality.at[index, 'type'] = 'Prioritas Tinggi'
        elif review > 100:
            gdf_hospitality.at[index, 'type'] = 'Prioritas Sedang'
        elif review <= 100:
            gdf_hospitality.at[index, 'type'] = 'Prioritas Rendah'
    
    gdf_hospitality_other = gdf_hospitality[gdf_hospitality['category'] == 'other']
    gdf_hotel = gdf_hospitality[gdf_hospitality['sub_category'] == 'Hotel']
    gdf_villa = gdf_hospitality[gdf_hospitality['sub_category'] == 'Villa']
    gdf_guest = gdf_hospitality[gdf_hospitality['sub_category'] == 'Guest House']
    gdf_board = gdf_hospitality[gdf_hospitality['sub_category'] == 'Boarding House']
    gdf_apart = gdf_hospitality[gdf_hospitality['sub_category'] == 'Apartement']

    compile_ = pd.concat([
        gdf_hospitality_other, gdf_hotel, gdf_villa, gdf_guest,  gdf_board, gdf_apart
        ], ignore_index=True)
    gdf_hospitality = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs)

    if verbose:
        master_list = [
            ['hotel', gdf_hotel, len(gdf_hotel)],
            ['villa', gdf_villa, len(gdf_villa)],
            ['guest', gdf_guest, len(gdf_guest)],
            ['board', gdf_board, len(gdf_board)],
            ['apart', gdf_apart, len(gdf_apart)]
        ]

        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((hotel, villa, guest, board, apart))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((hotel, villa, guest, board, apart))

        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result is Empty!')
        print(f'Total Hotel POI Classified : {result_summary['hotel']}')
        print(f'Total Villa POI Classified : {result_summary['villa']}')
        print(f'Total Guest House POI Classified : {result_summary['guest']}')
        print(f'Total Boarding House POI Classified : {result_summary['board']}')
        print(f'Total Apartement POI Classified : {result_summary['apart']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ Hospitality and Accomodation POI Filtering Finished!')

    if verbose:
        if result_gdf is not None: 
            return result_gdf.reset_index(drop=True)
    else: 
        print(f'\n✅ Hospitality and Accomodation POI Filtering Finished!')
        return gdf_hospitality.reset_index(drop=True)

def poi_industry(
    gdf_industry, 
    cleaned_name_column, 
    subclass_column, 
    verbose=True,
    factory=True,
    warehouse=True,
    office=True,
    q1_review = 5, 
    q3_review = 50):

    print('⚙️ Processing Industry Category Point of Interest (POI) Filter Object')
    gdf_industry[cleaned_name_column] = gdf_industry[cleaned_name_column].apply(clean_name)
    gdf_industry[subclass_column] = gdf_industry[subclass_column].apply(clean_name)
    gdf_ = len(gdf_industry)

    word_office = ['office', 'group', 'global', 'corp', 'property', 'technopark', 'menara',
        'tower', 'indonesia', 'business', 'griya', 'kantor', 'properti', 'official',
        'asos', 'pln', 'pnm']

    factory, warehouse, office = validate_subclass({
        'factory' : factory, 'warehouse' : warehouse, 'office' : office}).values()

    for index, row in tqdm(gdf_industry.iterrows(), total = len(gdf_industry)):
        name = row[cleaned_name_column]
        sub = row[subclass_column]
        review = int(row['review'])

        best_match_bank, score_bank = correct_typo_dict(name, BANKING_BANK_BRAND, threshold=85)

        # if any(potential in name for potential in INDUSTRY_FACTORY_KEY) and any(
        #     potential in sub for potential in INDUSTRY_FACTORY_KEY):
        #     gdf_industry.at[index, 'category'] = 'Industry'
        #     gdf_industry.at[index, 'sub_category'] = 'Factory'
        #     gdf_industry.at[index, 'score'] = 0
        #     gdf_industry.at[index, 'log'] = '✅ | Match'
        
        if any(potential in name for potential in INDUSTRY_FACTORY_KEY):
            gdf_industry.at[index, 'category'] = 'Industry'
            gdf_industry.at[index, 'sub_category'] = 'Factory'
            gdf_industry.at[index, 'score'] = 0
            gdf_industry.at[index, 'log'] = '✅ | Match Factory Name'

        elif any(potential in name for potential in INDUSTRY_WAREHOUSE_KEY):
            gdf_industry.at[index, 'category'] = 'Industry'
            gdf_industry.at[index, 'sub_category'] = 'Warehouse'
            gdf_industry.at[index, 'score'] = 0
            gdf_industry.at[index, 'log'] = '✅ | Match Warehouse Name'

        elif any(potential in sub for potential in INDUSTRY_OFFICE_KEY) and 'cv' in name:
            gdf_industry.at[index, 'category'] = 'Industry'
            gdf_industry.at[index, 'sub_category'] = 'Office'
            gdf_industry.at[index, 'score'] = 0
            gdf_industry.at[index, 'log'] = '✅ | Contain CV'
        
        elif any(potential in sub for potential in INDUSTRY_OFFICE_KEY) and any(
            potential in name for potential in word_office):
            gdf_industry.at[index, 'category'] = 'Industry'
            gdf_industry.at[index, 'sub_category'] = 'Office'
            gdf_industry.at[index, 'score'] = 0
            gdf_industry.at[index, 'log'] = '✅ | Match Office Sus Contain Emerge Word'
    
        elif not any(noise in name for noise in BANKING_SUB) and not any(
                noise in name for noise in HEALTH_SUB) and not any(
                    noise in name for noise in STORE_SUB) and not any(
                        noise in name for noise in UNIV_SUB) and not any(
                            noise in name for noise in HOSPITALITY_SUB): 
            
            if any(potential in name for potential in INDUSTRY_OFFICE_KEY):
                if score_bank >= 85:
                    gdf_industry.at[index, 'category'] = 'other'
                    gdf_industry.at[index, 'log'] = '❌ | Bank POI Alert, Not Office'
                    continue
                else:
                    gdf_industry.at[index, 'category'] = 'Industry'
                    gdf_industry.at[index, 'sub_category'] = 'Office'
                    gdf_industry.at[index, 'score'] = 0
                    gdf_industry.at[index, 'log'] = '✅ | Match Office'
            
            elif review > 4 and any(potential in sub for potential in INDUSTRY_OFFICE_KEY):
                gdf_industry.at[index, 'category'] = 'Industry'
                gdf_industry.at[index, 'sub_category'] = 'Office'
                gdf_industry.at[index, 'score'] = 0
                gdf_industry.at[index, 'log'] = '✅ | Match Office Sus, But review > 4'
            
            else:
                gdf_industry.at[index, 'category'] = 'other'
                gdf_industry.at[index, 'log'] = '❌ | Match Subclass | Else'
        
        else:
            gdf_industry.at[index, 'category'] = 'other'
            gdf_industry.at[index, 'log'] = '❌  | Else'

        if review < q1_review: gdf_industry.at[index, 'type'] = 'Prioritas Rendah'
        elif review <q3_review: gdf_industry.at[index, 'type'] = 'Prioritas Sedang'
        else: gdf_industry.at[index, 'type'] = 'Prioritas Tinggi'

    gdf_industry_other = gdf_industry[gdf_industry['sub_category'] == 'other']
    gdf_factory = gdf_industry[gdf_industry['sub_category'] == 'Factory']
    gdf_warehouse = gdf_industry[gdf_industry['sub_category'] == 'Warehouse']
    gdf_office = gdf_industry[gdf_industry['sub_category'] == 'Office']

    compile_ = pd.concat([gdf_industry_other, gdf_factory, gdf_warehouse, gdf_office], ignore_index=True)
    gdf_industry = gpd.GeoDataFrame(compile_, geometry='geometry', crs=compile_.crs)

    if verbose:
        master_list = [
            ['factory', gdf_factory, len(gdf_factory)],
            ['warehouse', gdf_warehouse, len(gdf_warehouse)],
            ['office', gdf_office, len(gdf_office)],
        ]

        merged_dictionary = merge_selected(master_list)
        result_gdf = merged_dictionary.get((factory, warehouse, office))

        result_dictionary = generate_result(master_list, gdf_)
        result_summary = result_dictionary.get((factory, warehouse, office))

        print('\nSummary') 
        print('-------') 
        if result_gdf is None : print(f'⚠️ GDF Result is Empty!')
        print(f'Total Factory POI Classified : {result_summary['factory']}')
        print(f'Total Warehouse POI Classified : {result_summary['warehouse']}')
        print(f'Total Office POI Classified : {result_summary['office']}')
        print(f"Total Classified : {result_summary['classified']}")
        print(f"Total Unclassified : {result_summary['unclassified']}")
        print(f'\n✅ Industry POI Filtering Finished!')

    if verbose:
        if result_gdf is not None: 
            return result_gdf.reset_index(drop=True)
    else: 
        print(f'\n✅ Industry POI Filtering Finished!')
        return gdf_industry.reset_index(drop=True)

In [66]:
word_office = ['office', 'group', 'global', 'corp', 'property', 'technopark', 'menara',
    'tower', 'indonesia', 'business', 'griya', 'kantor', 'properti', 'official',
    'asos', 'pln', 'pnm']

word_office = ','.join(word_office)
word_office

'office,group,global,corp,property,technopark,menara,tower,indonesia,business,griya,kantor,properti,official,asos,pln,pnm'

In [ ]:
BANKING_BANK_BRAND
bank_brand = BANKING_BANK_BRAND.copy()
bank_brand = ','.join(bank_brand.values())

In [ ]:
gdf = gpd.read_parquet(rf'C:\Users\Ahmad Haikal\Documents\SPIE 2.0\Data\POI Indo 400K.parquet')

gdf['cleaned_name'] = gdf['name'].apply(clean_name)
gdf['subclass'] = gdf['subclass'].apply(clean_name) 
gdf = refine_crs(gdf)

gdf = set_value(gdf)

describe_gdf(gdf)

In [ ]:
# TDRAFT KEYWORD POI FNB & TEMP FUNCTION

def get_top_word(gdf, column_name):
    series = gdf[column_name].dropna().astype(str)
    all_words = ' '.join(series).split()
    word_counts = Counter(all_words)
    common_words = word_counts.most_common(500)
    df = pd.DataFrame(common_words, columns=['name', 'count'])
    print(df)
    return(df)

def pattern_data(gdf, column, list_word):
    pattern = '|'.join(list_word)
    gdf_new = gdf[gdf[column].str.contains(pattern)]
    return gdf_new

pattern_keyword_resto = [
    'meal', 'restaurant', 'bistro', 'pusat kuliner', 'resto', 'rumah makan', 
    'tempat kuliner', 'tempat makan', 'warung makan', 'tourist', 'food court']

pattern_keyword_fastfood = [
    'kfc', 'mcdonald', 'mcd', 'subway', 'wingstop', 'hokben', 'hoka hoka bento', 'burger king',
    'wendys', 'pizza hut', 'texas chicken', 'papa john', 'marugame udon', 'ichiban sushi', 
    'krispy kreme', 'jco', 'dunkin', 'richeese', 'cfc', 'solaria', 'gacoan', 'shihlin', 'delivery'
    ]

# FNB_SUB_1 = [
#     'kfc', 'mcdonald', 'mcd', 'subway', 'wingstop', 'hokben', 'hoka hoka bento', 'burger king',
#     'wendys', 'pizza hut', 'texas chicken', 'papa john', 'marugame udon', 'ichiban sushi', 
#     'krispy kreme', 'jco', 'dunkin', 'richeese', 'cfc', 'solaria', 'gacoan', 'shihlin', 'delivery'
#     ]

#----------------
FNB_FAST_SUB = [
    'belanja', 'makanan', 'restoran', 'ayam', 'pizza', 'sandwich', 'siap saji', 'kopi', 'jepang', 'donat',

]

FNB_FAST_KEY = [
    'kfc ', ' kfc','mcdonald', 'mcd ', ' mcd','subway', 'wingstop', 'hokben', 'hoka hoka bento', 'burger king',
    'wendys', 'pizza hut', 'texas chicken', 'papa john', 'marugame udon', 'ichiban sushi', 
    'krispy kreme', 'jco', 'dunkin', 'richeese', 'cfc', 'solaria', 'gacoan', 'shihlin', 
    ]
#----------------

pattern_keyword_bakery = ['bakery']

pattern_keyword_bar = ['bar', 'liquor']

CAFE_SUB = ['kafe', 'kedai kopi']

CAFE_SUB_NOISE = ['toko', 'hotel', 'pasar', 'pasok', 'pemasok', 'pabrik', 'hostel', 'villa' ]
CAFE_KEY = [
    'kafe', 'kupi', 'kopi', 'cafe', 'caffe', 'caf', 'brew', 'coffee', 'coffe', 'cofee',
    'roastery', 'janji jiwa', 'kopi kenangan', 'fore', 'tomoro',
]
CAFE_KEY_NOISE = ['warung', 'pasar', 'pasok', 'pemasok', 'pabrik', 'warkop', 'warong', 'burjo']

FNB_BAKERY_SUB = ['toko kue', 'toko roti', 'cookie', 'pastri', 'hidangan penutup']
FNB_BAKERY_KEY = ['bakery', 'cake', 'holland bakery', 'bread', 'patisserie', 'mako']

FNB_BAR_SUB = [
    'bar ', ' bar','minuman keras', 'minuman anggur','alkohol', 'pub ', ' pub', 'gastropub', 
    'toko bir', 'wine', 'koktail']

FNB_BAR_KEY = ['tipsy', 'liquor', 'vinyard', 'cocktail']

FNB_RESTO_KEY = ['bistro', ]

FNB_LOCAL_KEY = [
    'khas', 'warung', 'angkringan', 'lapo', 'kedai', 'warteg', 'warung makan','nasi goreng', 
    'nasi uduk', 'nasi padang', 'nasi campur', 'nasi pecel', 'nasi kuning', 'lontong sayur', 
    'soto ayam', 'soto betawi', 'soto kudus', 'soto madura', 'rawon', 'sate', 'bakso', 'mie ayam', 
    'mie goreng', 'bubur ayam', 'ayam geprek', 'pecel lele', 'ayam penyet', 'tahu gejrot', 
    'gado-gado', 'rujak', 'sop buntut', 'es teler', 'es campur', 'es doger', 'es dawet', 'wedang jahe', 
    'wedang ronde', 'serabi', 'klepon', 'lupis', 'cenil', 'onde-onde', 'martabak', 'terang bulan',
    'seblak', 'cilok', 'cireng', 'bakso bakar', 'sosis bakar', 'roti bakar', 'pisang bakar',
    'otak-otak', 'pempek', 'siomay', 'batagor', 'kebab', 'sate taichan', 'bakmi'
]

FNB_LOCAL_SUB = [
    'ayam', 'bakso', 
]

In [ ]:
# DRAFT POI FNB

def poi_fnb(gdf_fnb, clenaed_name_column, subclass_column, review_column):
    gdf_fnb[clenaed_name_column] = gdf_fnb[clenaed_name_column].apply(clean_name)
    gdf_fnb[subclass_column] = gdf_fnb[subclass_column].apply(clean_name)

    for index, row in tqdm(gdf_fnb.iterrows(), total=len(gdf_fnb)):
        name = row[clenaed_name_column]
        sub = row[subclass_column]
        review = row[review_column]

        match_coffee, score_coffee = correct_typo_token(name, CAFE_KEY)

        if any(potential in name for potential in FNB_FAST_KEY):
            if any(potential in sub for potential in FNB_FAST_SUB):
                if review > 10 :
                    gdf_fnb.at[index, 'category'] = 'Food and Beverage'
                    gdf_fnb.at[index, 'sub-category'] = 'Fast Food'
                    gdf_fnb.at[index, 'type'] = None
                    gdf_fnb.at[index, 'log'] = f'✅ | Match Name | Match Subclass'
                else:
                    gdf_fnb.at[index, 'category'] = 'Food and Beverage'
                    gdf_fnb.at[index, 'sub-category'] = 'Fast Food'
                    gdf_fnb.at[index, 'type'] = None
                    gdf_fnb.at[index, 'log'] = f'⚠️ | Match Name | Low Review'

            else:
                gdf_fnb.at[index, 'category'] = 'Food and Beverage'
                gdf_fnb.at[index, 'sub-category'] = 'Fast Food'
                gdf_fnb.at[index, 'type'] = None
                gdf_fnb.at[index, 'log'] = f'⚠️ | Match Name | Wrong Subclass'
        
        elif any(potential in sub for potential in CAFE_SUB):
            if any(noise in sub for noise in CAFE_SUB_NOISE):
                gdf_fnb.at[index, 'category'] = 'other'
                gdf_fnb.at[index, 'log'] = '❌ | Contain Sub Noise'
    
            elif review <= 50:
                if score_coffee > 90:
                    if any(noise in name for noise in CAFE_KEY_NOISE):
                        gdf_fnb.at[index, 'category'] = 'other'
                        gdf_fnb.at[index, 'log'] = '❌ | Match Subclass | Contain Name Noise'
                    else:
                        gdf_fnb.at[index, 'category'] = 'Food and Beverage'
                        gdf_fnb.at[index, 'sub-category'] = 'Coffee Shop'
                        gdf_fnb.at[index, 'type'] = None
                        gdf_fnb.at[index, 'log'] = f'✅ | Match Subclass | Match Name'

                else:
                    gdf_fnb.at[index, 'category'] = 'other'
                    gdf_fnb.at[index, 'log'] = '❌ | Match Subclass | Fake Cafffe Alert'

            else:
                gdf_fnb.at[index, 'category'] = 'other'
                gdf_fnb.at[index, 'log'] = '❌ | Match Subclass | Unmatch Name'
        
        elif any(potential in sub for potential in FNB_BAR_SUB):
            if any(potential in name for potential in FNB_BAR_KEY):
                gdf_fnb.at[index, 'category'] = 'Food and Beverage'
                gdf_fnb.at[index, 'sub-category'] = 'Bar'
                gdf_fnb.at[index, 'type'] = None
                gdf_fnb.at[index, 'log'] = f'✅ | Match Subclass | Match Name'
            
            else:
                gdf_fnb.at[index, 'category'] = 'Food and Beverage'
                gdf_fnb.at[index, 'sub-category'] = 'Bar'
                gdf_fnb.at[index, 'type'] = None
                gdf_fnb.at[index, 'log'] = f'⚠️| Match Subclass | Unmatch Name'

        else:
            gdf_fnb.at[index, 'category'] = 'other'
            # gdf_fnb.at[index, 'log'] = f'❌ | Wrong Name | Wrong Subclass'

    gdf_fnb = gdf_fnb[gdf_fnb['log'].notna()]
    return gdf_fnb

In [ ]:
restaurant_sub = [
    'restoran', 'kafe', 'rumah makan', 'pujasera', 'tegal', 'padang', 'indonesia', 
    'brunch', 'bistro', 'sesuai untuk keluarga', 'sarapan', 'jepang', 'masakan asia', 'jawa', 
    'hidangan laut'
]

In [ ]:
gdf.groupby('keyword').size().sort_values(ascending=False)
potential_fnb = gdf[gdf['keyword'].isin([
    'food and beverage', 'restaurant', 'cafe', 'bakery', 'fast food', 'bar', 'tempat makan', 'meal', 'bistro', 
    'pusat kuliner', 'resto', 'rumah makan', 'warung makan', 'food court', 'diner', 'kafe', 'tempat kuliner', 
    'warteg (warung tegal)', 
    ])]
word_potential = potential_fnb.groupby('subclass').size().sort_values(ascending=False) 
# word_potential = pd.DataFrame(word_potential).reset_index()
word_potential
# word_potential = get_top_word(word_potential, 'subclass')
# word_potential = word_potential['name'].tolist()
# # potential_list = ','.join(potential_list)

# word_potential

In [ ]:
# POI FNB

gdf_fnb = gdf.copy()
gdf_fnb = poi_fnb(gdf_fnb, 'cleaned_name', 'subclass', 'review') 
gdf_fnb = gdf_fnb[gdf_fnb['key'] == '']

In [ ]:
#POI FNB CHECK
fnb_cek = gdf_fnb[['name', 'subclass', 'review', 'sub-category', 'log']]

In [ ]:
# RESULT TEST FOR POI FNB

gdf_fnb_bar = gdf_fnb[gdf_fnb['sub-category'] == 'Bar']
gdf_fnb_bar = gdf_fnb_bar[['name', 'subclass', 'review', 'log']]

In [ ]:
# RECENT DRAFT FOR RESTO

gdf_fnb = gdf.copy()
FAST_FOOD_SUB  = ['']
RESTAURANT_SUB = ['restoran', 'padang', 'rumah makan', 'brunch', 'restaurant', 'eatery']
RESTAURANT_SUB_NOISE = [
    'toko', 'hotel', 'pasar', 'pasok', 'pemasok', 'pabrik', 'hostel', 'villa', 'kantin', 'losmen', 
    ]
RESTAURANT_KEY = ['restoran', 'rumah makan', 'rm ']
RESTAURANT_KEY_NOISE = [
    'warung', 'pasar', 'pasok', 'pemasok', 'pabrik', 'warkop', 'warong', 'burjo', 'hotel',
    'hostel', 'villa'
    ]


FAST_FOOD_POTENTIAL = ['delivery']
RESTAURANT_POTENTIAL = []

for index, row in tqdm(gdf_fnb.iterrows(), total=len(gdf_fnb)):
    name = row['cleaned_name']
    sub = row['subclass']
    review = int(row['review'])

    if not any(noise in sub for noise in RESTAURANT_SUB_NOISE):
        if any(potential in name for potential in RESTAURANT_KEY):
            if not any(noise in name for noise in RESTAURANT_KEY_NOISE):
                if any(potential in sub for potential in RESTAURANT_SUB):
                    gdf_fnb.at[index, 'category'] = 'Food and Beverage'
                    gdf_fnb.at[index, 'sub_category'] = 'Restaurant'
                    gdf_fnb.at[index, 'type'] = None
                    gdf_fnb.at[index, 'score'] = 100
                    gdf_fnb.at[index, 'log'] = '✅ | Match Name | Match Subclass'
                    continue

                else:
                    gdf_fnb.at[index, 'category'] = 'other'
                    gdf_fnb.at[index, 'log'] = '❌ | Match Name | Unmatch Subclass'
                    continue
            
            else:
                gdf_fnb.at[index, 'category'] = 'other'
                gdf_fnb.at[index, 'log'] = '❌ | Contain Noise in Name'
                continue
        
        elif any(potential in sub for potential in RESTAURANT_SUB):
            gdf_fnb.at[index, 'category'] = 'Food and Beverage'
            gdf_fnb.at[index, 'sub_category'] = 'Restaurant'
            gdf_fnb.at[index, 'type'] = None
            gdf_fnb.at[index, 'score'] = 100
            gdf_fnb.at[index, 'log'] = '✅ | Match Subclass | Alerta'
            continue

        else:
            gdf_fnb.at[index, 'category'] = 'other'
            gdf_fnb.at[index, 'log'] = '❌ | Match Name | Wrong Subclass'
            continue
    
    elif any(noise in sub for noise in RESTAURANT_SUB_NOISE):
        gdf_fnb.at[index, 'category'] = 'other'
        gdf_fnb.at[index, 'log'] = '❌ | Contain Noise in Subclass'

    # else:
    #     gdf_fnb.at[index, 'category'] = 'other'
    #     # gdf_fnb.at[index, 'log'] = '❌ | Else'
    #     continue shit kenapa gw cuma bengang bengong fak lah,

gdf_fnb = gdf_fnb[gdf_fnb['category'] != 'other']
gdf_fnb_test = gdf_fnb[['name', 'subclass', 'review', 'log']] 


In [ ]:
# RECENT DRAFT FOR FAST FOOD

gdf_fastfood = gdf.copy()

initiate = [
    'kfc', 'mcdonald', 'mcd', 'subway', 'wingstop', 'hokben', 'hoka hoka bento', 'burger king',
    'wendys', 'pizza hut', 'texas chicken', 'papa john', 'marugame udon', 'ichiban sushi', 
    'krispy kreme', 'jco', 'dunkin', 'richeese', 'cfc', 'solaria', 'gacoan', 'shihlin', 'delivery'
    ]

for index, row in tqdm(gdf_fastfood.iterrows(), total=len(gdf_fastfood)):
    name = row['cleaned_name']
    sub = row['subclass']
    review = int(row['review'])

    if any(potential in name for potential in initiate):
        gdf_fastfood.at[index, 'category'] = 'Food and Beverage'
        gdf_fastfood.at[index, 'sub_category'] = 'Fast Food'
        gdf_fastfood.at[index, 'type'] = None
        gdf_fastfood.at[index, 'score'] = 100
        gdf_fastfood.at[index, 'log'] = '✅ | Initiate Name Match'
    
    else:
        gdf_fastfood.at[index, 'category'] = 'other'
        gdf_fastfood.at[index, 'log'] = '❌ | Else'

gdf_fastfood = gdf_fastfood[gdf_fastfood['category'] != 'other']

In [ ]:
gdf_bankings = gdf.copy()

for index, row in tqdm(gdf_bankings.iterrows(), total=len(gdf_bankings)):
    name = row['cleaned_name']
    subclass = row['subclass']    
    best_match, score, brand_name = match_brand(name, BANKING_BANK_BRAND, threshold=70)


    if any(potential in subclass for potential in BANKING_SUB):
        if any(potential in name for potential in BANKING_ATM_KEY):
            if score > 75:
                gdf_bankings.at[index, 'category'] = 'Banking'
                gdf_bankings.at[index, 'sub_category'] = 'ATM'
                gdf_bankings.at[index, 'type'] = brand_name
                gdf_bankings.at[index, 'score'] = score
                gdf_bankings.at[index, 'log'] = '✅ ATM Match | Brand Match'
            else:
                gdf_bankings.at[index, 'category'] = 'Banking'
                gdf_bankings.at[index, 'sub_category'] = 'ATM'
                gdf_bankings.at[index, 'type'] = 'other'
                gdf_bankings.at[index, 'score'] = score
                gdf_bankings.at[index, 'log'] = '✅ ATM Match | Local Bank'
        
        elif name.startswith(tuple(BANKING_BANK_BI)):
            gdf_bankings.at[index, 'category'] = 'Banking'
            gdf_bankings.at[index, 'sub_category'] = 'Bank'
            gdf_bankings.at[index, 'type'] = 'Bank Indonesia'
            gdf_bankings.at[index, 'score'] = score
            gdf_bankings.at[index, 'log'] = '✅ ATM Match | Brand Match'
        
        elif score > 90:
            gdf_bankings.at[index, 'category'] = 'Banking'
            gdf_bankings.at[index, 'sub_category'] = 'Bank'
            gdf_bankings.at[index, 'type'] = brand_name
            gdf_bankings.at[index, 'score'] = score
            gdf_bankings.at[index, 'log'] = '✅ Bank | Brand Match > 90'
        
        else:
            if score > 75:
                gdf_bankings.at[index, 'category'] = 'Banking'
                gdf_bankings.at[index, 'sub_category'] = 'Bank'
                gdf_bankings.at[index, 'type'] = 'other'
                gdf_bankings.at[index, 'score'] = score
                gdf_bankings.at[index, 'log'] = '✅ Bank | Local Indicate > 75'
            
            elif name.startswith(tuple(BANKING_POTENTIAL)):
                gdf_bankings.at[index, 'category'] = 'Banking'
                gdf_bankings.at[index, 'sub_category'] = 'Bank'
                gdf_bankings.at[index, 'type'] = 'other'
                gdf_bankings.at[index, 'score'] = score
                gdf_bankings.at[index, 'log'] = '✅ Bank | Startwith Potential'
            
            elif any(potential in name for potential in BANKING_BANK_START):
                gdf_bankings.at[index, 'category'] = 'Banking'
                gdf_bankings.at[index, 'sub_category'] = 'Bank'
                gdf_bankings.at[index, 'type'] = 'other'
                gdf_bankings.at[index, 'score'] = score
                gdf_bankings.at[index, 'log'] = '✅ Bank | Startwith Start'

            else:
                gdf_bankings.at[index, 'category'] = 'Banking'
                gdf_bankings.at[index, 'sub_category'] = 'Microservices and Agent'
                gdf_bankings.at[index, 'type'] = 'other'
                gdf_bankings.at[index, 'score'] = score
                gdf_bankings.at[index, 'log'] = '✅ Else | Microservices'
    
    else:
        gdf_bankings.at[index, 'category'] = 'Else'
        gdf_bankings.at[index, 'log'] = '⚠️ Else'


In [ ]:
check = gdf_bankings[['name', 'subclass', 'category', 'sub_category', 'type', 'score','log']]

check_unclassified = check[
    check['category'].isna() & check['subclass'].str.contains('|'.join(BANKING_SUB))
    ]

In [ ]:
gdf

In [ ]:
# FINANCIAL SIMULATION



In [ ]:
gdf_finance = gdf.copy()

for index, row in tqdm(gdf_finance.iterrows(), total=len(gdf_finance)):
    name = row['cleaned_name']
    sub = row['subclass']

    if any(potential in sub for potential in FIN_SUB):
        if any(potential in name for potential in LIFE_KEY):
            gdf_finance.at[index, 'category'] = 'Financial Service'
            gdf_finance.at[index, 'sub_category'] = 'Insurance'
            gdf_finance.at[index, 'type'] = None
            gdf_finance.at[index, 'score'] = 100
            gdf_finance.at[index, 'log'] = '✅ | Ins | by Name | Match Name | Match Subclass'
        
        elif not any(potential in name for potential in LIFE_KEY) and 'asuransi' in sub:
            gdf_finance.at[index, 'category'] = 'Financial Service'
            gdf_finance.at[index, 'sub_category'] = 'Insurance'
            gdf_finance.at[index, 'type'] = None
            gdf_finance.at[index, 'score'] = 100
            gdf_finance.at[index, 'log'] = '✅ | Ins | by Name | Match Name | Match Subclass '
        
        elif name.startswith(tuple(COP_KEY)):
            gdf_finance.at[index, 'category'] = 'Financial Service'
            gdf_finance.at[index, 'sub_category'] = 'Cooperation'
            gdf_finance.at[index, 'type'] = None
            gdf_finance.at[index, 'score'] = 100
            gdf_finance.at[index, 'log'] = '✅ | Wrong Subclass | Start With Cop Name'
        
        elif any(potential in name for potential in PAWN_KEY):
            gdf_finance.at[index, 'category'] = 'Financial Service'
            gdf_finance.at[index, 'sub_category'] = 'Pawn Shop'
            gdf_finance.at[index, 'type'] = None
            gdf_finance.at[index, 'score'] = 100
            gdf_finance.at[index, 'log'] = '✅ | Pawn | by Name | Match Name | Match Subclass'
        
        else:
            gdf_finance.at[index, 'category'] = 'other'
            gdf_finance.at[index, 'log'] = '❌ | Wrong Name | Match Subclass'
    
    else:    
        if name.startswith(tuple(PAWN_KEY)):
            gdf_finance.at[index, 'category'] = 'Financial Service'
            gdf_finance.at[index, 'sub_category'] = 'Pawn Shop'
            gdf_finance.at[index, 'type'] = None
            gdf_finance.at[index, 'score'] = 100
            gdf_finance.at[index, 'log'] = '✅ | Wrong Subclass | Start With Pawn Name'
        
        elif name.startswith(tuple(COP_KEY)):
            gdf_finance.at[index, 'category'] = 'Financial Service'
            gdf_finance.at[index, 'sub_category'] = 'Cooperation'
            gdf_finance.at[index, 'type'] = None
            gdf_finance.at[index, 'score'] = 100
            gdf_finance.at[index, 'log'] = '✅ | Wrong Subclass | Start With Cop Name'
            
        elif any(potential in name for potential in LIFE_KEY):
            gdf_finance.at[index, 'category'] = 'other'
            gdf_finance.at[index, 'log'] = '❌ | Ins | Match Name | Wrong Subclass'

        elif any(potential in name for potential in COP_KEY):
            gdf_finance.at[index, 'category'] = 'other'
            gdf_finance.at[index, 'log'] = '❌ | Cop | Match Name | Wrong Subclass'

        elif any(potential in name for potential in PAWN_KEY):
            gdf_finance.at[index, 'category'] = 'other'
            gdf_finance.at[index, 'log'] = '❌ | Pawn | Match Name | Wrong Subclass'
        
        else:
            gdf_finance.at[index, 'category'] = 'other'
            gdf_finance.at[index, 'log'] = '❌ | Else '

gdf_finance_cek = gdf_finance[['name', 'subclass', 'sub_category', 'type','score', 'log']]


In [61]:
text = """
rs 
rumah sakit
rsud
rsu
rsia
rsk
rspi
rsj
rsd
rsbh
rsgm
rsab 
rsa 
rsad 
rsai 
rsal 
rsau 
rsb 
rsi 
rspmd 
rst 
rspad 
rspau 
rspal 
rsppn 
rsp 
rstp 
rsia 
rsnu 
rsmh 
rsmn 
rsmp 
pusat jantung
pusat nasional kanker
rscm
rscs
rsijt
rsit
rsmc
rspd
rspg
rspj
rspur
rssp
rswu

"""

result = ",".join(
    line.strip('\n')
    for line in text.splitlines()
    if line.strip()
)

print(result)

rs ,rumah sakit,rsud,rsu,rsia,rsk,rspi,rsj,rsd,rsbh,rsgm,rsab ,rsa ,rsad ,rsai ,rsal ,rsau ,rsb ,rsi ,rspmd ,rst ,rspad ,rspau ,rspal ,rsppn ,rsp ,rstp ,rsia ,rsnu ,rsmh ,rsmn ,rsmp ,pusat jantung,pusat nasional kanker,rscm,rscs,rsijt,rsit,rsmc,rspd,rspg,rspj,rspur,rssp,rswu


In [ ]:
gdfbanking = gdf.copy() 
gdfbanking = poi_bank(gdfbanking, 'cleaned_name', 'subclass', verbose = True, atm = True, bank = True, micro = True)
gdf_banking_cek = gdfbanking[['name', 'subclass', 'sub_category', 'type','score', 'log']]

In [ ]:
mandiri = gdf_banking_cek[gdf_banking_cek['log'].fillna('').str.contains('Mandiri')]
mandiri_suspect = mandiri[~mandiri['subclass'].isin(['atm', 'bank'])]

In [ ]:
micro = gdf_banking_cek[gdf_banking_cek['sub_category'] == 'Agent and Microservices'].groupby('subclass').size().sort_values(ascending=False)   
micro

In [ ]:
cek_bank = gdf_banking_cek = gdfbanking[['name', 'subclass', 'review','sub_category', 'type','score', 'log']]

In [ ]:
# TRIAL AND ERROR FOR BANKING POI

gdfbank_unc = gdf_banking_cek[gdf_banking_cek['sub_category'].isna()]
gdfbank_c = gdf_banking_cek[gdf_banking_cek['sub_category'].notna()]
# gdfbank_unc.groupby('subclass').size().sort_values(ascending=False)

pattern_banking = '|'.join(BANKING_SUB)

gdfban_unc_bank = gdfbank_unc[gdfbank_unc['name'].str.contains(pattern_banking)]

In [ ]:
gdffinance = gdf.copy()
gdffinance = poi_finance(
    gdffinance, 'cleaned_name', 'subclass', verbose = True, life = True, cop = True, pawn = True)
gdf_finance_cek = gdffinance[['name', 'subclass', 'sub_category', 'log']] 

In [ ]:
sub_dist = gdf_finance_cek[gdf_finance_cek['sub_category'].notna()].groupby('subclass').size().sort_values(ascending=False) 
sub_dist

In [ ]:
gdf_finance_cek[gdf_finance_cek['name'].groupby()]
for k, v in tdqm(gdffinance.iterrows(), total = len(gdffinance)):
    if v

In [ ]:
# TRIAL AND ERROR FOR FINANCIAL SERVICES POI

gdffin_unc = gdf_finance_cek[gdf_finance_cek['sub_category'].isna()]
gdffin_c = gdf_finance_cek[gdf_finance_cek['sub_category'].notna()]
# gdfbank_unc.groupby('subclass').size().sort_values(ascending=False)

pattern_fin = '|'.join(FIN_SUB)
gdfban_unc_fin = gdffin_unc[gdffin_unc['subclass'].str.contains(pattern_fin)]

In [ ]:
gdfhealth = gdf.copy() 
gdfhealth = poi_health(
    gdfhealth, 'cleaned_name', 'subclass', 'review', verbose=False ,hospital=True, clinic=True, pharmacy=True)
gdfh_health_cek = gdfhealth[['name', 'subclass', 'sub_category', 'review', 'log']]
positive_health = gdfh_health_cek[gdfh_health_cek['sub_category'].notna()]  

In [ ]:
# TRIAL AND ERROR FOR FINANCIAL SERVICES POI

gdfhealth_unc = gdfh_health_cek[gdfh_health_cek['sub_category'].isna()]
gdfhealth_c = gdfh_health_cek[gdfh_health_cek['sub_category'].notna()]
# gdfbank_unc.groupby('subclass').size().sort_values(ascending=False)

pattern_health = '|'.join(HEALTH_SUB)
gdfban_unc_health = gdfhealth_unc[gdfhealth_unc['subclass'].str.contains(pattern_health)]

In [ ]:
gdfschool = gdf.copy()
gdfschool = poi_school(
    gdfschool, 'cleaned_name', 'subclass', verbose=False, pre_school = True, elementary_school = True, 
    junior_high_school= True, senior_high_school= True, international = True, special = True, non_formal=True, unit=True
    )
gdfh_school_cek = gdfschool[['name', 'subclass', 'sub_category', 'type','score', 'log']]
positive_school = gdfh_school_cek[gdfh_school_cek['sub_category'].notna()]

In [ ]:
spesific = positive_school[positive_school['sub_category'] == 'Techincal Implementation Unit'].groupby('subclass').size().sort_values(ascending=False)
spesific

In [ ]:
pre_sub = positive_school[positive_school['sub_category'] == 'Pre-school'].groupby('subclass').size().sort_values(ascending=False)
pre_sub 

In [ ]:
pattern = '|'.join(SCHOOL_SUB)

unc = gdfh_school_cek[gdfh_school_cek['sub_category'].isna()]
unc = unc[unc['subclass'].str.contains(pattern)]

In [ ]:
# TRIAL AND ERROR FOR SCHOOL POI

gdfschool_unc = gdfh_school_cek[gdfh_school_cek['sub_category'].isna()]
gdfschool_c = gdfh_school_cek[gdfh_school_cek['sub_category'].notna()]
# gdfbank_unc.groupby('subclass').size().sort_values(ascending=False)

pattern_school = '|'.join(SCHOOL_SUB)
gdfban_unc_school = gdfschool_unc[gdfschool_unc['subclass'].str.contains(pattern_school)]

In [ ]:
gdfuniv = gdf.copy()
gdfuniv = poi_univ(gdfuniv, 'cleaned_name', 'subclass', 'review', verbose=False,
    university=True, institute=True, polytechnic=True, college=True, academy=True)
gdfh_univ_cek = gdfuniv[['name', 'subclass', 'sub_category', 'type','score', 'log']]
univ_positive = gdfh_univ_cek[gdfh_univ_cek['sub_category'].notna()]

In [ ]:
academy = univ_positive[univ_positive['sub_category'] == 'Academy']
college = univ_positive[univ_positive['sub_category'] == 'College']
institute = univ_positive[univ_positive['sub_category'] == 'Institute'] 
polytechnic = univ_positive[univ_positive['sub_category'] == 'Polytechnic']
university = univ_positive[univ_positive['sub_category'] == 'University']   

In [ ]:
gdfh_univ_cek = gdfuniv[['name', 'subclass', 'sub_category', 'type','score', 'log']]
name_count = gdfh_univ_cek[gdfh_univ_cek['sub_category'].notna()]
name_count['name'] = name_count['name'].apply(clean_name)
get_top_word(name_count, 'name')

In [ ]:
# TRIAL AND ERROR FOR UNIV POI

gdfuniv_unc = gdfh_univ_cek[gdfh_univ_cek['sub_category'].isna()]
gdfuniv_c = gdfh_univ_cek[gdfh_univ_cek['sub_category'].notna()]
# gdfbank_unc.groupby('subclass').size().sort_values(ascending=False)

pattern_univ = '|'.join(UNIV_SUB)
gdfban_unc_univ = gdfuniv_unc[gdfuniv_unc['subclass'].str.contains(pattern_univ)]

for index, row in gdfduniv 

In [ ]:
gdfstore = gdf.copy()
gdfstore = poi_shopping(
    gdfstore, 'cleaned_name', 'subclass', 'review', review_threshold=300, verbose=True,
    small=True, convenience=True, supermarket=True, mall=True)
gdfh_store_cek = gdfstore[['name', 'subclass', 'sub_category', 'type','score', 'log']]
positive = gdfh_store_cek[gdfh_store_cek['sub_category'].notna()]

In [ ]:
small = positive[positive['sub_category'] == 'Small Store']
small.groupby('name').size().sort_values(ascending=False)
get_top_word(small, 'name')

In [ ]:
# MY DIARY

i still doesnt know what i looking for
eugene story just his picture while running waering catchy sport outfit pointing at the camera with smiling face 

p info adu ModuleNotFoundErr

In [ ]:
# MY FEELINGS

so i intend to write my head whelm to this python notebook

you know i really horny right now, i want to burst!
its really hard to hold it you now,
but at the same time i want it, 

you know how its feel you tried hard to reach the surface from the abyss of lust 
while you got pulled down so hard, like if you relax your breath a second you will finally

i lazy code right now, i lazy learn for now,
what i think is about stockbit and stock, also the illegal lust app
i also want to check out my storage box, but i still thinking about it 

no one here surounding me, sometimes this is good, but i feel lonely as well

so right now i seeing stock graph li


----- 

you now that feeling alyaws come again, i feel like i dont know i can survive this well or not
i have big dream, wise, and glorious.
but i always fail and drop literally at the first step, fisrt challange
you know i want to escape this life, this unproductive, unwanted, and unhonored one
i dont know why i always want to escape, 

escape from parents
escape from rk 
escape from campus
escape from friends
escape from office
escape from cabinet 
escape from past

chat gpt say escapeing from anything mean you just lost yourself
or actually you want to escaping curent version of 
and that person just need one honest reason to live


In [ ]:
# MY FEELINGS
hari ini gw kayaknya oversleep
kemarin malam habis makan nasi + lauk buatan mama
(ada sempol udang ayam, ayam goreng, timun) + risol dan bolu tante uli

gw tidur kecepetan kayaknya dan bangunnya pun masih agak kesiangan 
(oversleep)

gw terus kepikiran sama aa alfy, gw mau bangun hubungan sehat tapi gimana ya


In [ ]:
gdfhospitality = gdf.copy()
gdfhospitality = poi_hospitality(
    gdfhospitality, 'cleaned_name', 'subclass', 'review', min_review=10, verbose=True,
    hotel=True, villa=True, guest=True, board=True, apart=True)
gdfh_hos_cek = gdfhospitality[['name', 'subclass', 'sub_category', 'type','score', 'log']] 
positive_hos = gdfh_hos_cek[gdfh_hos_cek['sub_category'].notna()]

In [ ]:
guest = positive_hos[positive_hos['sub_category'] == 'Apartement'].groupby('subclass').size().sort_values(ascending=False)
guest

In [67]:
gdfindustry = gdf.copy()
gdfindustry = poi_industry(
    gdfindustry, 'cleaned_name', 'subclass', verbose=True, 
    factory=True, warehouse=True, office=True, q1_review=5, q3_review=50) 
gdfh_ind_cek = gdfindustry[['name', 'subclass', 'sub_category', 'type','score', 'log']]

⚙️ Processing Industry Category Point of Interest (POI) Filter Object


100%|██████████| 495112/495112 [02:21<00:00, 3498.44it/s]



Summary
-------
Total Factory POI Classified : 301
Total Warehouse POI Classified : 576
Total Office POI Classified : 19468
Total Classified : 20345
Total Unclassified : 474767

✅ Industry POI Filtering Finished!


In [ ]:
compile =  pd.concat(
    [gdfbanking, gdffinance, gdfhealth, gdfschool, gdfstore, gdfuniv, gdfhospitality, gdfindustry], ignore_index=True
    )


compile = compile.drop(columns='score')
poi_classified = gpd.GeoDataFrame(compile, geometry='geometry', crs=compile.crs)

poi = gdf.copy()
# poi_classified = poi_classified.drop_duplicates(subset='name', keep='first')
poi_unclassified = poi[~poi['name'].isin(poi_classified['name'])]

poi = refine_crs(poi)
poi_classified = refine_crs(poi_classified)
poi_unclassified = refine_crs(poi_unclassified)


In [64]:
unc = poi_unclassified.groupby('subclass').size().sort_values(ascending=False)
unc_office = poi_unclassified[poi_unclassified['subclass'] == 'kantor perusahaan']


In [ ]:
sum_clas = len(poi_classified)
sum_unclas = len(poi_unclassified)
print(f'Total Classified POI: {sum_clas}')
print(f'Total Unclassified POI: {sum_unclas}')

In [ ]:
dupes = poi_classified[poi_classified.duplicated(subset=['latitude', 'longitude'], keep=False)]

In [ ]:
poi_unclassified.groupby('subclass').size().sort_values(ascending=False)

In [ ]:
poi_unclassified.count()

In [ ]:
toko = poi_unclassified[poi_unclassified['subclass'].str.contains('toko')]
toko.groupby('subclass').size().sort_values(ascending=False)

In [ ]:
export_dir = f"Result/{datetime.now().strftime('%Y%m%d')}_Cleaned POI"
if not os.path.exists(export_dir):
    os.makedirs(export_dir) 

poi.to_parquet(os.path.join(export_dir, 'POI Data'))
poi_classified.to_parquet(os.path.join(export_dir, 'POI Classified'))
poi_unclassified.to_parquet(os.path.join(export_dir, 'POI Unclassified'))

grouped = poi_classified.groupby(
    ["Island", "Provinsi", "Kabkot", 'Kecamatan', "category", 'sub_category']).size().reset_index(name='count')
grouped_index = grouped.set_index(
    ["Island", "Provinsi", "Kabkot", 'Kecamatan', "category", 'sub_category']).unstack(["category", "sub_category"], fill_value=0)
grouped_index.to_excel(os.path.join(export_dir, "POI Summary.xlsx"))


In [ ]:
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool

In [ ]:
df_labeled = gpd.read_parquet(rf'C:\Users\Ahmad Haikal\Downloads\Draft 4 POI\Result\20251110_Cleaned POI\POI Classified')
df_unlabeled = gpd.read_parquet(rf'C:\Users\Ahmad Haikal\Downloads\Draft 4 POI\Data\POI Bali Jawa.parquet')

cat_features = ['name', 'subclass']
num_features = ['latitude', 'longitude']

# Model Category
target_cat = 'category'
X = df_labeled[cat_features + num_features]
y = df_labeled[target_cat]

train_pool = Pool(X, y, cat_features=cat_features)
model_cat = CatBoostClassifier(iterations=400, depth=8, learning_rate=0.1, loss_function='MultiClass', verbose=100)
model_cat.fit(train_pool)

X_unlabeled = df_unlabeled[cat_features + num_features]
df_unlabeled['pred_category'] = model_cat.predict(X_unlabeled).ravel()

# Model Sub-Category
target_sub = 'sub_category'
X2 = df_labeled[cat_features + num_features + [target_cat]]
y2 = df_labeled[target_sub]

train_pool2 = Pool(X2, y2, cat_features=cat_features + [target_cat])
model_sub = CatBoostClassifier(
    iterations=400, depth=8, learning_rate=0.1, loss_function='MultiClass', verbose=100)
model_sub.fit(train_pool2)

df_unlabeled['pred_sub_category'] = model_sub.predict(
    df_unlabeled[cat_features + num_features + ['pred_category']]
)

# Model Type
target_type = 'type'
X3 = df_labeled[cat_features + num_features + [target_cat, target_sub]]
y3 = df_labeled[target_type]

train_pool3 = Pool(X3, y3, cat_features=cat_features + [target_cat, target_sub])
model_type = CatBoostClassifier(iterations=400, depth=8, learning_rate=0.1, loss_function='MultiClass', verbose=100)
model_type.fit(train_pool3)

# Prediksi type ke unlabeled
df_unlabeled['pred_type'] = model_type.predict(
    df_unlabeled[cat_features + num_features + ['pred_category', 'pred_sub_category']]
)

df_unlabeled.to_csv("poi_unlabeled_predicted.csv", index=False)
print("✅ Selesai! Hasil prediksi disimpan ke poi_unlabeled_predicted.csv")

In [ ]:
from catboost import CatBoostClassifier, Pool
import geopandas as gpd

# ===== Load data =====
df_labeled = gpd.read_parquet(r'C:\Users\Ahmad Haikal\Downloads\Draft 4 POI\Result\20251110_Cleaned POI\POI Classified')
df_unlabeled = gpd.read_parquet(r'C:\Users\Ahmad Haikal\Downloads\Draft 4 POI\Data\POI Bali Jawa.parquet')

# ===== Feature setup =====
cat_features = ['name', 'subclass']
num_features = ['latitude', 'longitude']
target = 'sub_category'

# Pastikan data labeled punya kolom target
if target not in df_labeled.columns:
    raise ValueError(f"❌ Kolom target '{target}' gak ditemukan di df_labeled.")

# ===== Train data =====
X_train = df_labeled[cat_features + num_features]
y_train = df_labeled[target]

train_pool = Pool(X_train, y_train, cat_features=cat_features)

# ===== Train model =====
model_sub = CatBoostClassifier(
    iterations=400,
    depth=8,
    learning_rate=0.1,
    loss_function='MultiClass',
    verbose=100
)
model_sub.fit(train_pool)

# ===== Predict on unlabeled =====
X_unlabeled = df_unlabeled[cat_features + num_features]
df_unlabeled['pred_sub_category'] = model_sub.predict(X_unlabeled).ravel().astype(str)

# ===== Save =====
df_unlabeled.to_parquet("poi_pred_subcategory_only.parquet", index=False)
print("✅ Selesai! Hasil prediksi sub_category disimpan ke poi_pred_subcategory_only.parquet")

# Optional preview
print(df_unlabeled[['name', 'pred_sub_category']].head(10))


In [ ]:
classified_count = poi['name'].count()
unclasified_count = gdf['name'].count() - classified_count
coverage = (classified_count / gdf['name'].count()) * 100

print('Summary')
print('-------')
print(f'Total Succesfully Filtered Data : {classified_count}')
print(f'Total Unclassified : {unclasified_count}')
print(f'Coverage : {coverage:.2f}%')
print(F'Accuracy : 49. 23%')

In [ ]:
# draft1
"""
POI RULE ENGINE (SIMPLIFIED & READABLE)
=====================================
Goal:
- Same use case
- Same behavior
- Less abstraction
- More explicit steps
- Easier to read, debug, and explain

This version sacrifices DRY-ness for clarity.
"""

import pandas as pd

# =====================================================
# 1. RULE FUNCTIONS (EXPLICIT & HUMAN-READABLE)
# =====================================================

def check_name_contains(poi, words):
    name = str(poi.get("name", "")).lower()
    for w in words:
        if w.lower() in name:
            return True
    return False


def check_name_startswith(poi, prefix):
    name = str(poi.get("name", "")).lower()
    return name.startswith(prefix.lower())


def check_review_gte(poi, threshold):
    return poi.get("review", 0) >= threshold


# =====================================================
# 2. FUNCTION MAP (FUNC STRING → FUNCTION)
# =====================================================

FUNC_MAP = {
    "name_contains": check_name_contains,
    "name_startswith": check_name_startswith,
    "review_gte": check_review_gte,
}


# =====================================================
# 3. LOAD CONFIG FILE
# =====================================================

def load_config_excel(path):
    return pd.read_excel(path)


# =====================================================
# 4. PARSE ONE RULE ROW (VERY EXPLICIT)
# =====================================================

def parse_one_rule(row):
    checks = []

    idx = 1
    while True:
        func_col = f"func{idx}"
        val_col = f"val{idx}"

        if func_col not in row:
            break

        if pd.isna(row[func_col]):
            break

        checks.append({
            "func": row[func_col],
            "val": row[val_col]
        })

        idx += 1

    return {
        "checks": checks,
        "category": row["category"],
        "subcategory": row.get("subcategory")
    }


# =====================================================
# 5. VALIDATE RULE (NO MAGIC)
# =====================================================

def is_rule_valid(rule):
    for c in rule["checks"]:
        if c["func"] not in FUNC_MAP:
            return False
    return True


# =====================================================
# 6. BUILD RULE ENGINE (READABLE VERSION)
# =====================================================

def build_rule_engine(config_df):
    rules = []

    for _, row in config_df.iterrows():
        rule = parse_one_rule(row)

        if not is_rule_valid(rule):
            continue

        executable_checks = []
        for c in rule["checks"]:
            func = FUNC_MAP[c["func"]]
            param = c["val"]
            executable_checks.append((func, param))

        rules.append({
            "checks": executable_checks,
            "result": (rule["category"], rule["subcategory"])
        })

    return rules


# =====================================================
# 7. LOAD POI DATA
# =====================================================

def load_poi_csv(path):
    return pd.read_csv(path)

# =====================================================
# 8. CLASSIFY POI (STEP-BY-STEP)
# =====================================================

def classify_poi_dataset(poi_df, rules):
    poi_df = poi_df.copy()
    poi_df["category"] = None
    poi_df["subcategory"] = None

    for idx, row in poi_df.iterrows():
        poi = row.to_dict()

        for rule in rules:
            rule_passed = True

            for func, param in rule["checks"]:
                if not func(poi, param):
                    rule_passed = False
                    break

            if rule_passed:
                poi_df.at[idx, "category"] = rule["result"][0]
                poi_df.at[idx, "subcategory"] = rule["result"][1]
                break

        if poi_df.at[idx, "category"] is None:
            poi_df.at[idx, "category"] = "UNCLASSIFIED"

    return poi_df


# =====================================================
# 9. MAIN PIPELINE (VERY CLEAR)
# =====================================================

def run_pipeline(config_path, poi_path):
    print("Loading config...")
    config_df = load_config_excel(config_path)

    print("Building rule engine...")
    rules = build_rule_engine(config_df)

    print("Loading POI data...")
    poi_df = load_poi_csv(poi_path)

    print("Classifying POI...")
    result_df = classify_poi_dataset(poi_df, rules)

    print("Done.")
    return result_df


# =====================================================
# Example
# =====================================================
# df = run_pipeline("config.xlsx", "poi.csv")
# df.to_csv("classified_poi.csv", index=False)


In [ ]:
class TextProcess(name) :
    def __init__(self):
        self.name = name